In [ ]:
import os
import sys
import glob
import subprocess
import importlib.metadata as metadata
from packaging.utils import parse_wheel_filename, canonicalize_name
from packaging.tags import sys_tags
import torch

CODE = '/kaggle/tmp/knee-mri-findings'
if torch.cuda.device_count() == 0:
    raise RuntimeError('no GPU: choose the accelerator GPU T4 x2')
comps, assets, check = [], None, None
for root, dirs, files in os.walk('/kaggle/input'):
    dirs[:] = sorted(d for d in dirs if d not in {'train_series', 'test_series', 'models', 'segmentation', 'wheels', 'training', 'crops'})
    if {'sample_submission.csv', 'test.csv'} & set(files) or os.path.isdir(f'{root}/test_series'):
        comps.append(root)
    if assets is None and os.path.isfile(f'{root}/models/blend.json'):
        assets = root
    if check is None and os.path.isdir(f'{root}/crops') and any(f.endswith('.npz') for f in os.listdir(f'{root}/crops')):
        check = f'{root}/crops'
comps.sort(key=lambda r: (os.path.basename(r) != 'rsna-knee-abnormality-detection', '/competitions/' not in r,
                          not os.path.isdir(f'{r}/test_series'), len(r)))
if not comps or assets is None:
    raise RuntimeError(f'attach the competition data and the knee-mri-assets output (found: data {comps[:1]}, assets {assets})')
data = comps[0]
if os.path.isdir(f'{assets}/training/crops'):
    check = f'{assets}/training/crops'
seg = [os.path.dirname(p) for p in glob.glob(f'{assets}/segmentation/*/*/plans.json')]
if len(seg) != 1:
    raise RuntimeError(f'expected one segmentation model in {assets}/segmentation, found {seg}')
tags = set(sys_tags())
wheels = {}
for w in sorted(glob.glob(f'{assets}/wheels/*.whl')):
    name, _, _, wtags = parse_wheel_filename(os.path.basename(w))
    wheels.setdefault(str(name), [])
    if wtags & tags:
        wheels[str(name)].append(w)
image = {canonicalize_name(d.metadata['Name']) for d in metadata.distributions() if d.metadata and d.metadata['Name']}
missing = [n for n, v in wheels.items() if not v and n not in image]
if missing:
    raise RuntimeError(f'Python {sys.version.split()[0]}: no wheel and not in the image: {missing}')
pick = [v[0] for v in wheels.values() if v]
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-index', '--no-deps', *pick], capture_output=True, text=True)
if r.returncode:
    raise RuntimeError('offline install failed: ' + (r.stderr or r.stdout)[-3000:])
r = subprocess.run([sys.executable, '-c', 'import nnunetv2, SimpleITK, pydicom, timm, cc3d, libjpeg, openjpeg, cv2'],
                   capture_output=True, text=True)
if r.returncode:
    raise RuntimeError('imports failed: ' + r.stderr[-3000:])
os.makedirs(f'{CODE}/knee', exist_ok=True)
os.chdir(CODE)
print(f'python {sys.version.split()[0]} | wheels {len(pick)} | GPUs {torch.cuda.device_count()} | data {data} | assets {assets} | '
      f'check crops {check}')

In [ ]:
%%writefile knee/__init__.py

In [ ]:
%%writefile knee/findings.py
import numpy as np

LABELS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis',
          "Baker's", 'Contusion', 'Fracture']

FOCUS = {
    'ACL': (['notch'], ['lateral_TF', 'medial_TF', 'joint']),
    'MCL': (['MCL'], ['medial_TF', 'bones']),
    'Medial Meniscus': (['medial_TF'], ['notch', 'MCL']),
    'Lateral Meniscus': (['lateral_TF'], ['notch']),
    'Medial OA': (['medial_TF'], ['bones', 'notch', 'MCL']),
    'Lateral OA': (['lateral_TF'], ['bones', 'notch']),
    'PF OA': (['PF'], ['joint']),
    'Effusion': (['joint', 'PF'], ['popliteal', 'notch']),
    'Synovitis': (['joint', 'PF'], ['notch', 'popliteal']),
    "Baker's": (['popliteal'], ['joint', 'medial_TF']),
    'Contusion': (['bones', 'medial_TF', 'lateral_TF'], ['PF', 'notch']),
    'Fracture': (['bones', 'medial_TF', 'lateral_TF', 'PF'], ['notch', 'MCL']),
}


def relevance(crop_names):
    R = np.full((len(LABELS), len(crop_names)), 2, np.int64)
    for i, label in enumerate(LABELS):
        main, near = FOCUS[label]
        for j, name in enumerate(crop_names):
            region = name.split('__')[0]
            R[i, j] = 0 if region in main else 1 if region in near else 2
    return R


def multilabel_folds(Y, k=5, seed=42):
    rng = np.random.default_rng(seed)
    Y = np.asarray(Y, bool)
    n, L = Y.shape
    fold = np.full(n, -1)
    need_l = np.tile(Y.sum(0) / k, (k, 1)).astype(float)
    need_f = np.full(k, n / k)
    left = np.ones(n, bool)
    while left.any():
        cnt = Y[left].sum(0)
        if cnt.max() == 0:
            for i in rng.permutation(np.where(left)[0]):
                f = int(np.argmax(need_f + rng.random(k) * 1e-06))
                fold[i] = f
                need_f[f] -= 1
                left[i] = False
            break
        col = int(np.argmin(np.where(cnt > 0, cnt, np.inf)))
        for i in rng.permutation(np.where(left & Y[:, col])[0]):
            f = int(np.lexsort((rng.random(k), need_f, need_l[:, col]))[-1])
            fold[i] = f
            need_l[f] -= Y[i]
            need_f[f] -= 1
            left[i] = False
    return fold

In [ ]:
%%writefile knee/series.py
import os
import re
import shutil
from concurrent.futures import ProcessPoolExecutor
import numpy as np
import pandas as pd
import pydicom

FS_TOKENS = {'FS', 'SAT', 'FSAT', 'FATSAT', 'STIR', 'SPIR', 'SPAIR', 'TIRM', 'SPECIAL', 'CHESS'}
FS_PARTS = ('FAT', 'WATER', 'DIXON', 'SPAIR', 'STIR', 'FATSAT', 'FS_', '_FS')
GRE_WORDS = ['FFE', 'GRE', 'SPGR', 'FLASH', 'MPRAGE', 'DESS', 'FIESTA', 'VIBE', 'WATS', 'MEDIC', 'TRUFI', 'MERGE', 'COSMIC']
HCOLS = ['n_slices', 'manufacturer', 'model', 'field', 'laterality', 'scanning_sequence', 'image_type', 'series_desc', 'mr_acq',
         'TR', 'TE', 'thickness', 'tsx', 'plane_hdr', 'fs_hdr', 'error']
STUDY_COLS = ['StudyInstanceUID', 'split', 'study_dir', 'lat_tag', 'site_key', 'ref_uid', 'ref_dir', 'alt_uid', 'alt_dir',
              'any_uid', 'any_dir']
PLANES = {'sag': 'Sagittal', 'cor': 'Coronal', 'axi': 'Axial', 'tra': 'Axial'}


def subdirs(p):
    return sorted(d for d in os.listdir(p) if os.path.isdir(f'{p}/{d}')) if os.path.isdir(p) else []


def read_header(sid, uid, d):
    files = [f for f in os.listdir(d) if f.endswith('.dcm')]
    if not files:
        raise FileNotFoundError(f'{sid} {uid}: no .dcm file in {d}')
    h = pydicom.dcmread(os.path.join(d, files[0]), stop_before_pixels=True)

    def g(k):
        return str(getattr(h, k, '')).strip()
    iop = h.get('ImageOrientationPatient')
    plane = None
    if iop is not None and len(iop) == 6:
        v = np.array([float(x) for x in iop])
        plane = ['Sagittal', 'Coronal', 'Axial'][int(np.argmax(np.abs(np.cross(v[:3], v[3:]))))]
    txt = (g('SeriesDescription') + ' ' + g('SequenceName') + ' ' + g('ScanOptions') + ' ' + g('ProtocolName')).upper()
    tok = set(txt.replace('-', ' ').replace('_', ' ').replace('/', ' ').replace('\\', ' ').split())
    fs = int(bool(tok & FS_TOKENS) or any(w in txt for w in FS_PARTS) or 'IR' in re.findall(r'[A-Z]+', g('ScanningSequence').upper()))
    return dict(StudyInstanceUID=sid, SeriesInstanceUID=uid, n_slices=len(files), manufacturer=g('Manufacturer'),
                model=g('ManufacturerModelName'), field=g('MagneticFieldStrength'), laterality=g('Laterality').upper(),
                scanning_sequence=g('ScanningSequence').upper(), image_type=g('ImageType').upper(),
                series_desc=g('SeriesDescription').upper(), mr_acq=g('MRAcquisitionType').upper(),
                TR=g('RepetitionTime'), TE=g('EchoTime'), thickness=g('SliceThickness'),
                tsx=h.file_meta.TransferSyntaxUID.name, plane_hdr=plane, fs_hdr=fs)


def header_row(job):
    sid, uid, d, strict = job
    try:
        return read_header(sid, uid, d)
    except Exception as e:
        if strict:
            raise RuntimeError(f'header of {d}: {e!r}') from e
        return dict(StudyInstanceUID=sid, SeriesInstanceUID=uid, n_slices=0, error=repr(e)[:200])


def mirror_study(job):
    sid, src, dst = job
    if os.path.isdir(dst):
        shutil.rmtree(dst)
    groups = {}
    for root, dirs, files in os.walk(src):
        dirs.sort()
        for f in sorted(files):
            p = os.path.join(root, f)
            try:
                u = str(pydicom.dcmread(p, stop_before_pixels=True, specific_tags=['SeriesInstanceUID']).SeriesInstanceUID)
            except Exception:
                continue
            groups.setdefault(u, []).append(p)
    rows = []
    for u, ps in groups.items():
        d = os.path.join(dst, u)
        os.makedirs(d, exist_ok=True)
        for i, p in enumerate(ps):
            os.symlink(p, os.path.join(d, f'{i:05d}.dcm'))
        rows.append(dict(StudyInstanceUID=sid, SeriesInstanceUID=u, series_dir=d))
    return rows


def series_rows(ids, base, csv, split):
    if os.path.exists(csv):
        C = pd.read_csv(csv, dtype={'StudyInstanceUID': str, 'SeriesInstanceUID': str})
    else:
        C = pd.DataFrame(columns=['StudyInstanceUID', 'SeriesInstanceUID'])
    C = C[C.StudyInstanceUID.isin(set(ids))].drop_duplicates(['StudyInstanceUID', 'SeriesInstanceUID'])
    C = C.drop(columns=[c for c in C.columns if c in HCOLS or c in ('split', 'series_dir')])
    listed = set(C.StudyInstanceUID)
    extra = pd.DataFrame([dict(StudyInstanceUID=s, SeriesInstanceUID=u) for s in ids if s not in listed for u in subdirs(f'{base}/{s}')],
                         columns=['StudyInstanceUID', 'SeriesInstanceUID'])
    S = pd.concat([C, extra], ignore_index=True) if len(extra) else C.reset_index(drop=True)
    S['split'] = split
    S['series_dir'] = [f'{base}/{s}/{u}' for s, u in zip(S.StudyInstanceUID, S.SeriesInstanceUID)]
    return S


def rebuild_folders(TS, ids, test_dir, folders, out):
    def has_dcm(d):
        return os.path.isdir(d) and any(f.endswith('.dcm') for f in os.listdir(d))
    ok = {s for s, g in TS[TS.split == 'test'].groupby('StudyInstanceUID') if any(has_dcm(d) for d in g.series_dir)}
    odd = [s for s in ids if s in folders and s not in ok]
    if not odd:
        return TS, 0
    with ProcessPoolExecutor(4) as pool:
        M = [r for rows in pool.map(mirror_study, [(s, f'{test_dir}/{s}', f'{out}/{s}') for s in odd]) for r in rows]
    M = pd.DataFrame(M, columns=['StudyInstanceUID', 'SeriesInstanceUID', 'series_dir'])
    M = M.merge(TS[TS.StudyInstanceUID.isin(set(odd))].drop(columns=['series_dir', 'split']),
                on=['StudyInstanceUID', 'SeriesInstanceUID'], how='left').assign(split='test')
    return pd.concat([TS[~TS.StudyInstanceUID.isin(set(odd))], M], ignore_index=True), M.StudyInstanceUID.nunique()


def add_headers(TS, strict):
    jobs = list(TS.assign(strict=strict)[['StudyInstanceUID', 'SeriesInstanceUID', 'series_dir', 'strict']].itertuples(index=False, name=None))
    with ProcessPoolExecutor(4) as pool:
        H = pd.DataFrame(list(pool.map(header_row, jobs, chunksize=16)))
    for c in HCOLS:
        if c not in H:
            H[c] = None
    SM = TS.merge(H[['StudyInstanceUID', 'SeriesInstanceUID'] + HCOLS], on=['StudyInstanceUID', 'SeriesInstanceUID'], how='left')
    for c in ('Anatomical_Plane', 'Fat_Suppression'):
        if c not in SM:
            SM[c] = None
    ap = SM.Anatomical_Plane.fillna('').astype(str).str.strip()
    SM['Anatomical_Plane'] = ap.str[:3].str.lower().map(PLANES).where(lambda x: x.notna(), ap.where(ap != '', SM.plane_hdr))
    words = {'true': 1, 'yes': 1, 'false': 0, 'no': 0}
    fs = pd.to_numeric(SM.Fat_Suppression.map(lambda v: words.get(str(v).strip().lower(), v)), errors='coerce')
    SM['Fat_Suppression'] = fs.where(fs.notna(), pd.to_numeric(SM.fs_hdr, errors='coerce')).fillna(0).astype(int)
    if 'Fluid_Sensitive' not in SM:
        SM['Fluid_Sensitive'] = SM.Fat_Suppression
    for c in ('manufacturer', 'model', 'field'):
        SM[c] = SM[c].fillna('').astype(str)
    SM['n_slices'] = pd.to_numeric(SM.n_slices, errors='coerce').fillna(0).astype(int)
    SM['site_key'] = SM.manufacturer + '|' + SM.model + '|' + SM.field
    return SM


def study_table(SM):
    SM = SM.copy()
    SM['lat_norm'] = SM.laterality.map({'R': 'R', 'L': 'L', 'RIGHT': 'R', 'LEFT': 'L'})
    sag = SM[(SM.Anatomical_Plane == 'Sagittal') & SM.error.isna()].copy()
    txt = sag.image_type.fillna('').astype(str) + ' ' + sag.series_desc.fillna('').astype(str)
    seq = sag.scanning_sequence.fillna('').astype(str)
    sag['is_gre'] = (seq.str.contains('GR') & ~seq.str.contains('SE')) | txt.map(lambda s: any(w in s for w in GRE_WORDS)).astype(bool)
    sag['is_3d'] = (sag.mr_acq.fillna('') == '3D') | (sag.n_slices >= 60)
    sag['odd_n'] = (sag.n_slices < 12) | (sag.n_slices > 50)
    sag = sag.sort_values(['StudyInstanceUID', 'is_gre', 'is_3d', 'Fat_Suppression', 'odd_n', 'n_slices'],
                          ascending=[True, True, True, True, True, False])
    sag['rank'] = sag.groupby('StudyInstanceUID').cumcount()
    refs = sag[sag['rank'] == 0].set_index('StudyInstanceUID')
    alts = sag[(sag['rank'] == 1) & ~sag.is_gre].set_index('StudyInstanceUID')
    order = {'Sagittal': 0, 'Coronal': 1, 'Axial': 2}
    rows = []
    for (sid, split), g in SM.groupby(['StudyInstanceUID', 'split'], sort=False):
        tags = g.lat_norm.dropna().unique()
        ok = g[g.error.isna()].sort_values('Anatomical_Plane', key=lambda s: s.map(order).fillna(3))
        rows.append(dict(StudyInstanceUID=sid, split=split, study_dir=os.path.dirname(g.series_dir.iloc[0]),
                         lat_tag=tags[0] if len(tags) == 1 else '', site_key=g.site_key.mode().iloc[0],
                         ref_uid=refs.SeriesInstanceUID.get(sid, ''), ref_dir=refs.series_dir.get(sid, ''),
                         alt_uid=alts.SeriesInstanceUID.get(sid, ''), alt_dir=alts.series_dir.get(sid, ''),
                         any_uid=ok.SeriesInstanceUID.iloc[0] if len(ok) else '', any_dir=ok.series_dir.iloc[0] if len(ok) else ''))
    return pd.DataFrame(rows, columns=STUDY_COLS)

In [ ]:
%%writefile knee/bones.py
import os
import json
from collections import Counter
import numpy as np
import pydicom
import SimpleITK as sitk
from scipy import ndimage as ndi

THREADS = int(os.environ.get('KNEE_THREADS', '1'))
sitk.ProcessObject.SetGlobalDefaultNumberOfThreads(THREADS)
BONES = {1: 'femur', 2: 'patella', 3: 'fibula', 4: 'tibia'}
ROIS = ['medial_TF', 'lateral_TF', 'notch', 'PF', 'MCL', 'joint', 'popliteal', 'bones']


def load_volume(d):
    dss = [pydicom.dcmread(os.path.join(d, f)) for f in os.listdir(d) if f.endswith('.dcm')]
    shape0 = Counter((int(x.Rows), int(x.Columns)) for x in dss).most_common(1)[0][0]
    dss = [x for x in dss if (int(x.Rows), int(x.Columns)) == shape0]
    iop = np.array(dss[0].ImageOrientationPatient, float)
    nrm = np.cross(iop[:3], iop[3:])
    proj = np.array([np.dot(np.array(x.ImagePositionPatient, float), nrm) for x in dss])
    o = np.argsort(proj)
    dss = [dss[i] for i in o]
    proj = proj[o]
    keep = np.r_[True, np.diff(np.round(proj, 2)) > 0]
    dss = [x for x, k in zip(dss, keep) if k]
    proj = proj[keep]
    px = [x.pixel_array for x in dss]
    if px[0].ndim != 2:
        raise ValueError(f'multi-frame pixel data {px[0].shape}')
    vol = np.stack(px).astype(np.float32)
    vol = vol * float(getattr(dss[0], 'RescaleSlope', 1) or 1) + float(getattr(dss[0], 'RescaleIntercept', 0) or 0)
    if getattr(dss[0], 'PhotometricInterpretation', '') == 'MONOCHROME1':
        vol = vol.max() - vol
    ps = [float(v) for v in dss[0].PixelSpacing]
    dz = float(np.median(np.diff(proj))) if len(proj) > 1 else float(getattr(dss[0], 'SliceThickness', 1.0) or 1.0)
    return vol, (dz, ps[0], ps[1]), iop, np.array(dss[0].ImagePositionPatient, float), dss[0]


def series_image(d):
    vol, (dz, dy, dx), iop, origin, hdr = load_volume(d)
    img = sitk.GetImageFromArray(vol)
    X, Y = iop[:3], iop[3:]
    img.SetDirection(np.stack([X, Y, np.cross(X, Y)], axis=1).flatten().tolist())
    img.SetSpacing((dx, dy, dz))
    img.SetOrigin(origin.tolist())
    return img, str(getattr(hdr, 'SeriesDescription', ''))


def to_las(img):
    return sitk.DICOMOrient(img, 'LAS')


def resample_to(img, new_sp, interp):
    old_sp, old_sz = np.array(img.GetSpacing()), np.array(img.GetSize())
    new_sz = np.maximum(1, np.round(old_sz * old_sp / np.array(new_sp))).astype(int).tolist()
    return sitk.Resample(img, new_sz, sitk.Transform(), interp, img.GetOrigin(), tuple(float(s) for s in new_sp),
                         img.GetDirection(), 0.0, img.GetPixelID())


def clean_mask(m):
    vox_ml = float(np.prod(m.GetSpacing())) / 1000.0
    arr = sitk.GetArrayFromImage(m)
    arr[arr > 4] = 0
    out, st = np.zeros_like(arr, dtype=np.uint8), {}
    min_vox = max(1, int(round(0.5 / vox_ml)))
    for lab, name in BONES.items():
        b = sitk.GetImageFromArray((arr == lab).astype(np.uint8))
        b.CopyInformation(m)
        cc = sitk.GetArrayFromImage(sitk.RelabelComponent(sitk.ConnectedComponent(b), minimumObjectSize=min_vox))
        n_all, n_keep = int((arr == lab).sum()), int((cc > 0).sum())
        st[f'{name}_ml'] = round(n_keep * vox_ml, 2)
        st[f'{name}_ncc'] = int(cc.max())
        st[f'{name}_lcc'] = round(float((cc == 1).sum()) / max(n_keep, 1), 3)
        st[f'{name}_specks_ml'] = round((n_all - n_keep) * vox_ml, 2)
        out[cc > 0] = lab
    o = sitk.GetImageFromArray(out)
    o.CopyInformation(m)
    return o, st


def lps_points(arr, img, lab, cap=120000):
    idx = np.argwhere(arr == lab)
    if len(idx) == 0:
        return None
    if len(idx) > cap:
        idx = idx[np.random.default_rng(0).choice(len(idx), cap, replace=False)]
    D = np.array(img.GetDirection()).reshape(3, 3)
    s = np.array(img.GetSpacing())
    o = np.array(img.GetOrigin())
    return o + idx[:, ::-1].astype(np.float64) * s @ D.T


def pc(a, q):
    return float(np.percentile(a, q))


def box(xa, xb, ya, yb, za, zb):
    return [min(xa, xb), max(xa, xb), min(ya, yb), max(ya, yb), min(za, zb), max(za, zb)]


def fov_center(img):
    return [float(v) for v in img.TransformContinuousIndexToPhysicalPoint(((np.array(img.GetSize()) - 1) / 2).tolist())]


def quality(q):
    n = len([w for w in str(q.get('flag_reasons') or '').split('|') if w])
    lcc = float(np.mean([q.get(f'{b}_lcc', 0.0) for b in ('femur', 'tibia', 'patella')]))
    return int(q.get('qc_ok', 0)), int(q.get('geom_ok', 0)), -n, lcc


def better(a, b):
    return quality(a) > quality(b)


def jdump(obj, path):
    json.dump(obj, open(path, 'w'), default=lambda o: o.item() if hasattr(o, 'item') else str(o))


def bone_regions(mask, st, tag):
    arr = sitk.GetArrayFromImage(mask)
    P = {n: lps_points(arr, mask, lab) for lab, n in BONES.items()}
    fem, pat, fib, tib = P['femur'], P['patella'], P['fibula'], P['tibia']
    C = {n: None if v is None else v.mean(0) for n, v in P.items()}
    q, why = {}, []
    q['femur_above_tibia'] = bool(fem is not None and tib is not None and C['femur'][2] > C['tibia'][2])
    q['patella_ant_femur'] = bool(pat is not None and fem is not None and C['patella'][1] < C['femur'][1])
    q['fibula_post_tibia'] = bool(fib is not None and tib is not None and C['fibula'][1] > C['tibia'][1])
    lf = None if fib is None or tib is None else 'L' if C['fibula'][0] > C['tibia'][0] else 'R'
    q['lat_fibula'], q['lat_tag'] = lf, tag
    q['lat_agree'] = None if lf is None or tag is None else bool(lf == tag)
    if lf and (tag is None or lf == tag):
        q['lat_final'], q['lat_source'] = lf, 'fibula'
    elif tag:
        q['lat_final'], q['lat_source'] = tag, 'tag'
    else:
        q['lat_final'], q['lat_source'] = None, 'unknown'
        why.append('lat_unknown')
    if st['femur_ml'] < 40:
        why.append('femur_small')
    if st['tibia_ml'] < 30:
        why.append('tibia_small')
    if st['patella_ml'] < 2:
        why.append('patella_small')
    if st['patella_ml'] > 45:
        why.append('patella_big')
    if st['fibula_ml'] < 0.5:
        why.append('fibula_missing')
    for b in ('femur', 'tibia', 'patella'):
        if st[f'{b}_ml'] > 0 and st[f'{b}_lcc'] < 0.9:
            why.append(f'{b}_fragmented')
    if not q['femur_above_tibia']:
        why.append('geom_femur_tibia')
    if not q['patella_ant_femur']:
        why.append('geom_patella')
    if not q['fibula_post_tibia']:
        why.append('geom_fibula')
    if q['lat_agree'] is False:
        why.append('lat_mismatch')
    q['flag_reasons'], q['qc_ok'] = '|'.join(why), int(not why)
    q['geom_ok'] = int(fem is not None and tib is not None and q['femur_above_tibia'])
    if not q['geom_ok']:
        return q
    s = 1.0 if q['lat_final'] == 'L' else -1.0
    m = -s
    zf, zt = pc(fem[:, 2], 5), pc(tib[:, 2], 95)
    zj = 0.5 * (zf + zt)
    top = tib[tib[:, 2] > zt - 15]
    top = top if len(top) > 200 else tib
    con = fem[fem[:, 2] < zf + 25]
    con = con if len(con) > 200 else fem
    x0, x1, y0, y1 = pc(top[:, 0], 2), pc(top[:, 0], 98), pc(top[:, 1], 2), pc(top[:, 1], 98)
    cx0, cx1, cy0, cy1 = pc(con[:, 0], 2), pc(con[:, 0], 98), pc(con[:, 1], 2), pc(con[:, 1], 98)
    if x1 - x0 > cx1 - cx0 + 15:
        x0, x1 = cx0, cx1
    xc, hw, ym = 0.5 * (x0 + x1), 0.5 * (x1 - x0), 0.5 * (y0 + y1)
    xe = xc + m * hw
    R = {'medial_TF': box(xc, xc + m * (hw + 8), y0 - 5, y1 + 5, zj - 18, zj + 22),
         'lateral_TF': box(xc, xc + s * (hw + 8), y0 - 5, y1 + 5, zj - 18, zj + 22),
         'notch': box(xc - 12, xc + 12, y0, y1, zj - 8, zj + 35),
         'MCL': box(xe - m * 10, xe + m * 18, ym - 22, ym + 18, zj - 55, zj + 45),
         'popliteal': box(xc + s * 10, xc + m * (hw + 20), cy1 - 15, cy1 + 45, zj - 50, zj + 40)}
    ya = min(y0, cy0) - 5
    if pat is not None and st['patella_ml'] >= 2:
        px0, px1, py0, py1, pz0, pz1 = [pc(pat[:, i], k) for i in range(3) for k in (2, 98)]
        R['PF'] = box(px0 - 8, px1 + 8, py0 - 6, py1 + 18, pz0 - 8, pz1 + 8)
        R['joint'] = box(min(x0, cx0) - 12, max(x1, cx1) + 12, min(py0, y0) - 10, cy1 + 12, zj - 25, max(pz1 + 35, zj + 60))
        R['bones'] = box(min(x0, cx0, px0) - 10, max(x1, cx1, px1) + 10, min(ya, py0 - 5), max(y1, cy1) + 5, zj - 60,
                         max(zj + 70, pz1 + 10))
    else:
        R['joint'] = box(min(x0, cx0) - 12, max(x1, cx1) + 12, cy0 - 25, cy1 + 12, zj - 25, zj + 70)
        R['bones'] = box(min(x0, cx0) - 10, max(x1, cx1) + 10, ya, max(y1, cy1) + 5, zj - 60, zj + 70)
    q['K'], q['rois'], q['knee_width_mm'] = [xc, ym, zj], R, round(2 * hw, 1)
    q['femur_win_ml'] = round(float(np.mean((fem[:, 2] >= zj) & (fem[:, 2] <= zj + 60))) * st['femur_ml'], 2)
    q['tibia_win_ml'] = round(float(np.mean((tib[:, 2] <= zj) & (tib[:, 2] >= zj - 60))) * st['tibia_ml'], 2)
    return q


def fix_labels(m, patella_max_ml=45.0, fibula_max_ml=30.0):
    arr = sitk.GetArrayFromImage(m).astype(np.uint8)
    arr[arr > 4] = 0
    spz = np.array(m.GetSpacing())[::-1]
    D = np.array(m.GetDirection()).reshape(3, 3)
    org = np.array(m.GetOrigin())
    spx = np.array(m.GetSpacing())
    vox_ml = float(np.prod(spx)) / 1000.0

    def phys(idx):
        return org + np.asarray(idx, float)[..., ::-1] * spx @ D.T
    nS = arr.shape[0]
    info = dict(bridged=0, patella_found=0, fibula_found=0, femur_top=0, tibia_bottom=0, shaft_added_ml=0.0)
    obj, nobj = ndi.label(arr > 0)
    out = np.zeros_like(arr)
    if nobj == 0:
        o = sitk.GetImageFromArray(out)
        o.CopyInformation(m)
        return o, info
    votes = np.bincount(obj.ravel().astype(np.int64) * 5 + arr.ravel(), minlength=(nobj + 1) * 5).reshape(nobj + 1, 5)
    votes[0] = 0
    f_obj = int(np.argmax(votes[:, 1])) if votes[:, 1].max() > 0 else 0
    tv = votes[:, 4].copy()
    t_in_f = int(tv[f_obj]) if f_obj else 0
    tv[f_obj] = 0
    t_obj = int(np.argmax(tv)) if tv.max() > 0 else 0
    F = obj == f_obj if f_obj else np.zeros(arr.shape, bool)
    if f_obj and t_in_f > 2 * (tv[t_obj] if t_obj else 0) and t_in_f > 0:
        info['bridged'] = 1
        sl = ndi.find_objects(F.astype(np.uint8))[0]
        Fb, Ab = F[sl], arr[sl]
        d_f = ndi.distance_transform_edt(~((Ab == 1) & Fb), sampling=spz)
        d_t = ndi.distance_transform_edt(~((Ab == 4) & Fb), sampling=spz)
        T = np.zeros(arr.shape, bool)
        T[sl] = Fb & (d_t < d_f)
        F = F & ~T
    else:
        T = obj == t_obj if t_obj else np.zeros(arr.shape, bool)

    def pts(mask, cap=60000):
        idx = np.argwhere(mask)
        if len(idx) > cap:
            idx = idx[np.random.default_rng(0).choice(len(idx), cap, replace=False)]
        return phys(idx) if len(idx) else None
    Pf, Pt = pts(F), pts(T)
    if Pf is not None:
        zf = float(np.percentile(Pf[:, 2], 5))
        cond = Pf[Pf[:, 2] < zf + 25]
        cond_y = float(np.median(cond[:, 1])) if len(cond) else float(np.median(Pf[:, 1]))
        info['femur_top'] = int(np.where(F.any(axis=(1, 2)))[0].max() >= 0.8 * (nS - 1))
    if Pt is not None:
        zt = float(np.percentile(Pt[:, 2], 95))
        prox = Pt[Pt[:, 2] > zt - 40]
        tcx, tcy = float(np.mean(prox[:, 0])), float(np.mean(prox[:, 1]))
        info['tibia_bottom'] = int(np.where(T.any(axis=(1, 2)))[0].min() <= 0.2 * (nS - 1))

    def best_piece(lab, rule, max_ml, min_ml):
        pcs, n = ndi.label(arr == lab)
        if n == 0:
            return None
        size = np.bincount(pcs.ravel())[1:] * vox_ml
        com = ndi.center_of_mass(np.ones(arr.shape, np.uint8), pcs, range(1, n + 1))
        boxes = ndi.find_objects(pcs)
        best = None
        for k in range(n):
            if not min_ml <= size[k] <= max_ml:
                continue
            c = phys(np.array(com[k]))
            zr = phys(np.array([[boxes[k][0].start, 0, 0], [boxes[k][0].stop - 1, 0, 0]]))[:, 2]
            if rule(c, float(zr.min()), float(zr.max())) and (best is None or size[k] > size[best]):
                best = k
        return None if best is None else pcs == best + 1
    PAT = FIB = None
    if Pf is not None:
        PAT = best_piece(2, lambda c, z0, z1: c[1] < cond_y - 10 and zf - 15 <= c[2] <= zf + 90, patella_max_ml, 1.0)
    if Pt is not None and Pf is not None:
        FIB = best_piece(3, lambda c, z0, z1: z1 < zf + 5 and c[2] < zt - 5 and c[1] > tcy - 5 and abs(c[0] - tcx) > 12,
                         fibula_max_ml, 0.3)

    def promote(piece, lab):
        o_id = int(obj[tuple(np.argwhere(piece)[0])])
        if o_id not in (f_obj, t_obj) and votes[o_id, lab] >= 0.5 * votes[o_id].sum():
            return obj == o_id
        return piece
    out[F] = 1
    out[T] = 4
    if PAT is not None:
        out[promote(PAT, 2)] = 2
        info['patella_found'] = 1
    if FIB is not None:
        out[promote(FIB, 3)] = 3
        info['fibula_found'] = 1
    used = {int(v) for v in np.unique(obj[out > 0]) if v}
    msp = spz[1:]

    def footprint(mask, upper):
        zs = np.where(mask.any(axis=(1, 2)))[0]
        if not len(zs):
            return None
        zmed = int(np.median(zs))
        part = mask[zmed:] if upper else mask[:zmed + 1]
        a, b = np.where(part.any(axis=0))
        return zmed, a.min(), a.max(), b.min(), b.max()
    fpF, fpT = footprint(out == 1, True), footprint(out == 4, False)
    for k, sl in enumerate(ndi.find_objects(obj), start=1):
        if sl is None or k in used:
            continue
        ca, cl = (sl[1].start + sl[1].stop - 1) / 2, (sl[2].start + sl[2].stop - 1) / 2
        if fpF and sl[0].start > fpF[0]:
            ea, el = 5 / msp[0], 5 / msp[1]
            if fpF[1] - ea <= ca <= fpF[2] + ea and fpF[3] - el <= cl <= fpF[4] + el:
                out[obj == k] = 1
                info['shaft_added_ml'] = info.get('shaft_added_ml', 0) + float((obj == k).sum()) * vox_ml
                continue
        if fpT and sl[0].stop - 1 < fpT[0]:
            sa, sb = 3 / msp[0], 3 / msp[1]
            if fpT[1] + sa <= ca <= fpT[2] - sa and fpT[3] + sb <= cl <= fpT[4] - sb:
                out[obj == k] = 4
                info['shaft_added_ml'] = info.get('shaft_added_ml', 0) + float((obj == k).sum()) * vox_ml
    for lab in (1, 4, 2, 3):
        for x in range(out.shape[2]):
            sl2 = out[:, :, x]
            if (sl2 == lab).any():
                fill = ndi.binary_fill_holes(sl2 == lab) & (sl2 == 0)
                sl2[fill] = lab
    zs1, zs4 = np.where((out == 1).any(axis=(1, 2)))[0], np.where((out == 4).any(axis=(1, 2)))[0]
    info['femur_top'] = int(len(zs1) > 0 and zs1.max() >= 0.8 * (nS - 1))
    info['tibia_bottom'] = int(len(zs4) > 0 and zs4.min() <= 0.2 * (nS - 1))
    info['shaft_added_ml'] = round(info['shaft_added_ml'], 2)
    changed = (arr > 0) & (out != arr)
    info['changed_ml'] = round(float(changed.sum()) * vox_ml, 2)
    info['deleted_ml'] = round(float(((arr > 0) & (out == 0)).sum()) * vox_ml, 2)
    o = sitk.GetImageFromArray(out)
    o.CopyInformation(m)
    return o, info

In [ ]:
%%writefile knee/segment.py
import os
import json
import time
import glob
import argparse
import numpy as np
import pandas as pd
import SimpleITK as sitk
import torch
from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor
from knee import bones

ap = argparse.ArgumentParser()
ap.add_argument('--table', required=True)
ap.add_argument('--model', required=True)
ap.add_argument('--out', required=True)
ap.add_argument('--shard', type=int, default=0)
ap.add_argument('--nshards', type=int, default=1)
ap.add_argument('--deadline', type=float, default=1e18)
ap.add_argument('--strict', type=int, default=1)
A = ap.parse_args()
torch.backends.cudnn.benchmark = True
torch.set_num_threads(bones.THREADS)
MASKS, QC = f'{A.out}/masks', f'{A.out}/qc'
for d in (MASKS, QC):
    os.makedirs(d, exist_ok=True)
plans = json.load(open(f'{A.model}/plans.json'))
sp = plans['configurations']['3d_fullres']['spacing']
TGT = [0.0] * 3
for i, t in enumerate(plans['transpose_forward']):
    TGT[2 - t] = float(sp[i])
TGT = tuple(TGT)
ISO_SCALE, STEP = 0.7, 1.0
fold = sorted(os.path.basename(f) for f in glob.glob(f'{A.model}/fold_*'))[0].replace('fold_', '')
cuda = torch.cuda.is_available()
predictor = nnUNetPredictor(tile_step_size=STEP, use_gaussian=True, use_mirroring=False, perform_everything_on_device=cuda,
                            device=torch.device('cuda', 0) if cuda else torch.device('cpu'), verbose=False,
                            verbose_preprocessing=False, allow_tqdm=False)
predictor.initialize_from_trained_model_folder(A.model, use_folds=(int(fold),) if fold.isdigit() else (fold,),
                                               checkpoint_name='checkpoint_final.pth')


def predict(las):
    geo = bones.resample_to(las, tuple(t / ISO_SCALE for t in TGT), sitk.sitkLinear)
    props = {'sitk_stuff': {'spacing': TGT, 'origin': geo.GetOrigin(), 'direction': geo.GetDirection()}, 'spacing': list(TGT)[::-1]}
    seg = predictor.predict_single_npy_array(sitk.GetArrayFromImage(geo)[None].astype(np.float32), props, None, None, False)
    si = sitk.GetImageFromArray(seg.astype(np.uint8))
    si.CopyInformation(geo)
    out = sitk.Resample(si, las, sitk.Transform(), sitk.sitkNearestNeighbor, 0, sitk.sitkUInt8)
    if cuda:
        torch.cuda.empty_cache()
    return out


def run_one(sid, d, uid, tag):
    t0 = time.time()
    img, desc = bones.series_image(d)
    las = bones.to_las(img)
    t_load = time.time() - t0
    m, st = bones.clean_mask(predict(las))
    q = bones.bone_regions(m, st, tag)
    q.update(st)
    q.update(StudyInstanceUID=sid, SeriesInstanceUID=uid, series_desc=desc, mode='iso', step=STEP,
             spacing=[float(v) for v in las.GetSpacing()], size=[int(v) for v in las.GetSize()], fov_center=bones.fov_center(las),
             t_load=round(t_load, 1), t_total=round(time.time() - t0, 1))
    return m, q


def attempt(sid, d, uid, tag, stage):
    try:
        m, q = run_one(sid, d, uid, tag)
        q['stage'] = stage
        return m, q
    except Exception as e:
        if A.strict:
            raise RuntimeError(f'{sid} {uid}: {e!r}') from e
        if cuda:
            torch.cuda.empty_cache()
        return None, dict(StudyInstanceUID=sid, SeriesInstanceUID=uid, qc_ok=0, geom_ok=0, flag_reasons='error', error=repr(e)[:300],
                          stage=stage)


TB = pd.read_csv(A.table, dtype=str, keep_default_na=False).sort_values('StudyInstanceUID').reset_index(drop=True)
TB = TB[TB.index % A.nshards == A.shard]
TB = TB[[not os.path.exists(f'{QC}/{s}.json') for s in TB.StudyInstanceUID]]
print(f'[segment {A.shard}] {len(TB)} studies', flush=True)
t0, n = time.time(), 0
for r in TB.itertuples(index=False):
    if time.time() > A.deadline:
        print(f'[segment {A.shard}] time limit after {n} studies', flush=True)
        break
    sid, tag = r.StudyInstanceUID, (r.lat_tag if r.lat_tag in ('L', 'R') else None)
    m, q = attempt(sid, r.ref_dir, r.ref_uid, tag, 'ref')
    if not q.get('qc_ok') and r.alt_dir:
        m2, q2 = attempt(sid, r.alt_dir, r.alt_uid, tag, 'alt')
        if m2 is not None and bones.better(q2, q):
            q2['ref_flags'] = q.get('flag_reasons')
            m, q = m2, q2
        else:
            q['alt_flags'] = q2.get('flag_reasons')
    if m is not None:
        sitk.WriteImage(m, f'{MASKS}/{sid}.nii.gz', True)
    q['site_key'] = r.site_key
    bones.jdump(q, f'{QC}/{sid}.json')
    n += 1
    if n % 25 == 0 or n == len(TB):
        el = (time.time() - t0) / n
        print(f'[segment {A.shard}] {n}/{len(TB)} | {el:.1f} s/study | {el * (len(TB) - n) / 60:.0f} min left', flush=True)
print(f'[segment {A.shard}] done: {n} studies in {(time.time() - t0) / 60:.1f} min', flush=True)

In [ ]:
%%writefile knee/regions.py
import os
import json
import time
import glob
import argparse
from collections import Counter
from concurrent.futures import ProcessPoolExecutor
from concurrent.futures.process import BrokenProcessPool
import numpy as np
import pandas as pd
import pydicom
import SimpleITK as sitk
from knee import bones

KEEP = ('StudyInstanceUID', 'SeriesInstanceUID', 'series_desc', 'mode', 'step', 'spacing', 'size', 'fov_center', 'site_key',
        'ref_flags', 'alt_flags')
ROI_COLS = ['StudyInstanceUID', 'roi', 'xmin', 'xmax', 'ymin', 'ymax', 'zmin', 'zmax', 'source', 'lat', 'lat_source', 'qc_ok',
            'bones_ok', 'mask_stage']
S = {}


def side_vector(j):
    return np.array([1.0 if j.get('lat_final') == 'L' else -1.0, 1.0, 1.0])


def build_atlas(region_jsons, bounds):
    clean = [j for j in region_jsons if j.get('qc_ok') == 1 and j.get('rois') and len(j['rois']) == len(bones.ROIS)]
    off, half, kf = {r: [] for r in bones.ROIS}, {r: [] for r in bones.ROIS}, []
    for j in clean:
        V, K = side_vector(j), np.array(j['K'])
        kf.append((K - np.array(j['fov_center'])) * V)
        for r in bones.ROIS:
            b = np.array(j['rois'][r])
            c = (b[[0, 2, 4]] + b[[1, 3, 5]]) / 2
            off[r].append((c - K) * V)
            half[r].append((b[[1, 3, 5]] - b[[0, 2, 4]]) / 2)
    return dict(studies=len(clean), knee_from_fov=np.median(kf, 0).tolist(), bounds=bounds,
                regions={r: dict(offset=np.median(off[r], 0).tolist(), half=np.median(half[r], 0).tolist()) for r in bones.ROIS})


def header_fov_center(d):
    hs = [pydicom.dcmread(os.path.join(d, f), stop_before_pixels=True) for f in os.listdir(d) if f.endswith('.dcm')]
    shape0 = Counter((int(h.Rows), int(h.Columns)) for h in hs).most_common(1)[0][0]
    hs = [h for h in hs if (int(h.Rows), int(h.Columns)) == shape0]
    iop = np.array(hs[0].ImageOrientationPatient, float)
    r, c = iop[:3], iop[3:]
    ipp = np.array([np.array(h.ImagePositionPatient, float) for h in hs])
    w = ipp @ np.cross(r, c)
    ps = [float(v) for v in hs[0].PixelSpacing]
    mid = (ipp[int(np.argmin(w))] + ipp[int(np.argmax(w))]) / 2
    return (mid + r * ps[1] * (shape0[1] - 1) / 2 + c * ps[0] * (shape0[0] - 1) / 2).tolist()


def setup(work, strict):
    st = pd.read_csv(f'{work}/meta/studies.csv', dtype=str, keep_default_na=False)
    S.update(work=work, strict=strict, studies=st,
             tags={s: (v if v in ('L', 'R') else None) for s, v in zip(st.StudyInstanceUID, st.lat_tag)})


def study(sid):
    W, strict, ST, tags = S['work'], S['strict'], S['studies'], S['tags']
    qf, mf = f'{W}/seg/qc/{sid}.json', f'{W}/seg/masks/{sid}.nii.gz'
    if strict and not os.path.exists(mf):
        raise RuntimeError(f'{sid}: no bone mask (no usable sagittal series, or the segmentation failed)')
    q0 = json.load(open(qf)) if os.path.exists(qf) else dict(StudyInstanceUID=sid, flag_reasons='no_segmentation')
    try:
        m1, info = bones.fix_labels(sitk.ReadImage(mf))
        m2, st = bones.clean_mask(m1)
        ext = tags.get(sid)
        q = bones.bone_regions(m2, st, None if info['fibula_found'] else ext)
        q.update(st)
        q.update(info)
        q.update(side_dicom=ext, side_ext_source='dicom' if ext else 'none',
                 side_disagree=int(bool(info['fibula_found'] and ext and q.get('lat_fibula') and q['lat_fibula'] != ext)))
        why = [w for w in q['flag_reasons'].split('|') if w and w not in ('fibula_missing', 'patella_small')]
        notes = []
        if info.get('shaft_added_ml', 0) > 0:
            why = [w for w in why if w not in ('femur_fragmented', 'tibia_fragmented')]
            notes.append('shaft_reattached')
        if not info['femur_top']:
            notes.append('short_femur')
        if not info['tibia_bottom']:
            notes.append('short_tibia')
        if q['side_disagree']:
            notes.append('side_disagree')
        if not info['patella_found']:
            why.append('patella_not_found')
        if not info['fibula_found']:
            why.append('fibula_not_found')
        q['flag_reasons'] = '|'.join(dict.fromkeys(why))
        q['qc_ok'] = int(not why)
        q['notes'] = '|'.join(notes)
        for k in KEEP:
            if k in q0:
                q[k] = q0[k]
        q['stage'] = q0.get('stage', 'ref')
        q['seg_flags'] = q0.get('flag_reasons', '')
        sitk.WriteImage(m2, f'{W}/masks/{sid}.nii.gz', True)
    except Exception as e:
        if strict:
            raise RuntimeError(f'{sid}: {e!r}') from e
        q = dict(StudyInstanceUID=sid, SeriesInstanceUID=q0.get('SeriesInstanceUID'), qc_ok=0, geom_ok=0, flag_reasons='error',
                 error=repr(e)[:200], site_key=q0.get('site_key'), fov_center=q0.get('fov_center'),
                 seg_flags=q0.get('flag_reasons', ''), changed_ml=0.0)
        if tags.get(sid):
            q['lat_final'], q['lat_source'] = tags[sid], 'tag'
    if not q.get('fov_center'):
        r = ST[ST.StudyInstanceUID == sid].iloc[0]
        for d, uid in ((r.ref_dir, r.ref_uid), (r.any_dir, r.any_uid)):
            if not d:
                continue
            if strict:
                q['fov_center'] = header_fov_center(d)
            else:
                try:
                    q['fov_center'] = header_fov_center(d)
                except Exception:
                    continue
            if not q.get('SeriesInstanceUID'):
                q['SeriesInstanceUID'] = uid
            break
    if not q.get('site_key'):
        q['site_key'] = ST.loc[ST.StudyInstanceUID == sid, 'site_key'].iloc[0]
    bones.jdump(q, f'{W}/regions/{sid}.json')
    return sid


def guarded(sid):
    if S['strict']:
        return study(sid)
    try:
        return study(sid)
    except Exception as e:
        bones.jdump(dict(StudyInstanceUID=sid, qc_ok=0, geom_ok=0, flag_reasons='error', error=repr(e)[:200]),
                    f'{S["work"]}/regions/{sid}.json')
        return sid


def run_studies(sids, nproc, strict, work, t0):
    done, remaining = set(), list(sids)
    while remaining:
        try:
            with ProcessPoolExecutor(nproc, initializer=setup, initargs=(work, strict)) as pool:
                for sid in pool.map(guarded, remaining, chunksize=1):
                    done.add(sid)
                    if len(done) % 250 == 0:
                        print(f'[regions] {len(done)}/{len(sids)} | {(time.time() - t0) / 60:.1f} min', flush=True)
            remaining = []
        except BrokenProcessPool:
            if strict:
                raise
            remaining = [x for x in remaining if x not in done]
            for x in remaining[:nproc]:
                try:
                    with ProcessPoolExecutor(1, initializer=setup, initargs=(work, strict)) as p1:
                        p1.submit(guarded, x).result()
                except BrokenProcessPool:
                    bones.jdump(dict(StudyInstanceUID=x, qc_ok=0, geom_ok=0, flag_reasons='error', error='worker died'),
                                f'{work}/regions/{x}.json')
                done.add(x)
            remaining = remaining[nproc:]


def add_flags(J, bounds):
    for j in J.values():
        why = [w for w in str(j.get('flag_reasons') or '').split('|') if w]
        for c, b in (('femur_win_ml', 'femur'), ('tibia_win_ml', 'tibia'), ('patella_ml', 'patella')):
            if j.get(c) is not None and j[c] > bounds[c][1]:
                why.append(f'{b}_big')
        if j.get('fibula_ml') is not None and j['fibula_ml'] > bounds['fibula_ml'][1]:
            j['notes'] = '|'.join([n for n in str(j.get('notes') or '').split('|') if n] + ['fibula_big'])
        w_ = j.get('knee_width_mm')
        if w_ is not None and not (bounds['knee_width_mm'][0] <= w_ <= bounds['knee_width_mm'][1]):
            why.append('knee_width_odd')
        why = list(dict.fromkeys(why))
        j['flag_reasons'], j['qc_ok'] = '|'.join(why), int(not why)
        j['bones_ok'] = int(j.get('geom_ok', 0) == 1 and not any(w.startswith(('femur', 'tibia', 'geom_femur', 'knee_width', 'error'))
                                                                 for w in why))
        j['patella_ok'] = int(j['bones_ok'] == 1 and not any(w.startswith(('patella', 'geom_patella')) for w in why))


def region_boxes(J, atlas):
    KF = np.array(atlas['knee_from_fov'])
    ATL = {r: (np.array(v['offset']), np.array(v['half'])) for r, v in atlas['regions'].items()}
    rows = []
    for sid, j in J.items():
        V, R = side_vector(j), (j.get('rois') or {})
        if j['bones_ok'] and j.get('K'):
            K, ks = np.array(j['K']), 'knee'
        elif j.get('fov_center'):
            K, ks = np.array(j['fov_center']) + KF * V, 'fov'
        else:
            continue
        for r in bones.ROIS:
            if j['bones_ok'] and r in R and (r != 'PF' or j['patella_ok']):
                b, src = R[r], 'bones'
            else:
                o, h = ATL[r]
                c = K + o * V
                b, src = [c[0] - h[0], c[0] + h[0], c[1] - h[1], c[1] + h[1], c[2] - h[2], c[2] + h[2]], f'atlas_{ks}'
            rows.append(dict(StudyInstanceUID=sid, roi=r, xmin=b[0], xmax=b[1], ymin=b[2], ymax=b[3], zmin=b[4], zmax=b[5],
                             source=src, lat=j.get('lat_final'), lat_source=j.get('lat_source'), qc_ok=j['qc_ok'],
                             bones_ok=j['bones_ok'], mask_stage=j.get('stage')))
    return pd.DataFrame(rows, columns=ROI_COLS)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--work', required=True)
    ap.add_argument('--anatomy', required=True)
    ap.add_argument('--nproc', type=int, default=4)
    ap.add_argument('--strict', type=int, default=1)
    A = ap.parse_args()
    W, t0 = A.work, time.time()
    for d in (f'{W}/masks', f'{W}/regions'):
        os.makedirs(d, exist_ok=True)
    sids = pd.read_csv(f'{W}/meta/studies.csv', dtype=str, keep_default_na=False).StudyInstanceUID.tolist()
    run_studies(sids, A.nproc, bool(A.strict), W, t0)
    atlas = json.load(open(A.anatomy))
    J = {os.path.basename(f)[:-5]: json.load(open(f)) for f in glob.glob(f'{W}/regions/*.json')}
    add_flags(J, atlas['bounds'])
    for sid, j in J.items():
        bones.jdump(j, f'{W}/regions/{sid}.json')
    RO = region_boxes(J, atlas)
    RO.to_csv(f'{W}/meta/rois.csv', index=False)
    Q = pd.DataFrame(list(J.values()))
    for c in ('SeriesInstanceUID', 'site_key'):
        if c not in Q:
            Q[c] = None
    Q.drop(columns=[c for c in ('rois', 'K', 'fov_center', 'spacing', 'size') if c in Q]).to_csv(f'{W}/meta/regions.csv', index=False)
    n_bones = int(Q.bones_ok.sum()) if 'bones_ok' in Q else 0
    print(f'[regions] done: {len(J)} studies in {(time.time() - t0) / 60:.1f} min | bone boxes for {n_bones}', flush=True)


if __name__ == '__main__':
    main()

In [ ]:
%%writefile knee/crops.py
import os
import io
import json
import zlib
import warnings
from types import SimpleNamespace
from collections import Counter
import numpy as np
import pandas as pd
import pydicom
import cv2
from scipy import ndimage as ndi
from PIL import Image, features

VERSION = 2
OUT_PX, MAX_SLICES, MIN_SLICES = 128, 24, 3
MAX_LARGE = 16
LARGE_ROIS = ('joint', 'bones', 'popliteal', 'MCL')
MARGIN_MM, SLAB_MM, THIN_MM = 4.0, 3.0, 2.0
MIN_SERIES_IMAGES = 8
ROIS = ['medial_TF', 'lateral_TF', 'notch', 'PF', 'MCL', 'joint', 'popliteal', 'bones']
SLOTS = ['sag_nf', 'sag_fs', 'cor_fs', 'cor_nf', 'ax_fs']
VIEW = {'sag_nf': 'sag', 'sag_fs': 'sag', 'cor_fs': 'cor', 'cor_nf': 'cor', 'ax_fs': 'ax'}
CROPS = [('medial_TF', 'sag_nf'), ('medial_TF', 'sag_fs'), ('medial_TF', 'cor_fs'),
         ('lateral_TF', 'sag_nf'), ('lateral_TF', 'sag_fs'), ('lateral_TF', 'cor_fs'),
         ('notch', 'sag_nf'), ('notch', 'sag_fs'), ('notch', 'cor_fs'),
         ('MCL', 'cor_fs'), ('MCL', 'ax_fs'),
         ('PF', 'ax_fs'), ('PF', 'sag_fs'), ('PF', 'sag_nf'),
         ('popliteal', 'ax_fs'), ('popliteal', 'sag_fs'),
         ('joint', 'sag_fs'), ('joint', 'ax_fs'),
         ('bones', 'cor_fs'), ('bones', 'sag_fs'), ('bones', 'cor_nf')]
CROP_NAMES = [f'{r}__{s}' for r, s in CROPS]
ANCHOR = {'sag_nf': 'medial_TF', 'sag_fs': 'bones', 'cor_fs': 'bones', 'cor_nf': 'bones', 'ax_fs': 'joint'}
GRE_WORDS = ('FFE', 'GRE', 'SPGR', 'FLASH', 'MPRAGE', 'DESS', 'FIESTA', 'VIBE', 'WATS', 'MEDIC', 'TRUFI', 'MERGE',
             'COSMIC', 'T2*', 'T2_STAR', 'HEMO')
DERIVED_WORDS = ('MPR', 'SECONDARY', 'PROJECTION', 'REFORMAT', 'MIP', 'SUBTRACT')
WEBP = bool(features.check('webp'))


def _col(df, name, default=''):
    return df[name] if name in df.columns else pd.Series([default] * len(df), index=df.index)


def rank_series(S):
    plane = _col(S, 'Anatomical_Plane').fillna('').astype(str).str.strip().str[:3].str.lower()
    fs = pd.to_numeric(_col(S, 'Fat_Suppression', 0), errors='coerce').fillna(0).astype(int)
    txt = (_col(S, 'image_type').fillna('').astype(str) + ' ' + _col(S, 'series_desc').fillna('').astype(str)).str.upper()
    seq = _col(S, 'scanning_sequence').fillna('').astype(str).str.upper()
    tr = pd.to_numeric(_col(S, 'TR', np.nan), errors='coerce')
    te = pd.to_numeric(_col(S, 'TE', np.nan), errors='coerce')
    n = pd.to_numeric(_col(S, 'n_slices', 0), errors='coerce').fillna(0)
    acq = _col(S, 'mr_acq').fillna('').astype(str).str.upper()
    err = _col(S, 'error').fillna('').astype(str).str.strip().str.len() > 0
    gre = (seq.str.contains('GR') & ~seq.str.contains('SE')) | txt.apply(lambda t: any(w in t for w in GRE_WORDS))
    der = txt.apply(lambda t: any(w in t for w in DERIVED_WORDS))
    is3d = (acq == '3D') | (n >= 60)
    t1 = (tr < 1000).values
    pdw = ((tr >= 1000) & (te <= 60)).values
    unk = tr.isna().values
    stir = seq.str.contains('IR').values
    out = {}
    for slot in SLOTS:
        want_plane = {'sag': 'sag', 'cor': 'cor', 'ax': 'axi'}[VIEW[slot]]
        m = ((plane == want_plane) & (fs == int(slot.endswith('_fs')))).values
        if not m.any():
            continue
        if slot == 'sag_nf':
            pref = np.where(pdw, 0, np.where(t1 | unk, 1, 2))
        elif slot == 'cor_nf':
            pref = np.where(t1, 0, np.where(pdw | unk, 1, 2))
        else:
            pref = np.where(stir, 1, 0)
        K = pd.DataFrame({'uid': S.SeriesInstanceUID.astype(str).values, 'err': err.values, 'gre': gre.values,
                          'der': der.values, 'is3d': is3d.values, 'pref': pref, 'few': (n < 10).values,
                          'neg_n': -n.values})[m]
        out[slot] = K.sort_values(['err', 'gre', 'der', 'is3d', 'pref', 'few', 'neg_n'], kind='stable').uid.tolist()
    return out


def _f(v, default=None):
    try:
        return float(v)
    except Exception:
        return default


def _seq0(ds, name):
    try:
        v = ds.get(name)
        return v[0] if v is not None and len(v) else None
    except Exception:
        return None


def _decode(ds, path):
    try:
        return ds.pixel_array, False
    except Exception:
        import SimpleITK as sitk
        return sitk.GetArrayFromImage(sitk.ReadImage(path)), True


def _frame_geom(ds, k, nf):
    if nf == 1 and 'ImagePositionPatient' in ds:
        ipp, iop, ps = ds.get('ImagePositionPatient'), ds.get('ImageOrientationPatient'), ds.get('PixelSpacing')
        slope, inter = _f(ds.get('RescaleSlope', 1), 1.0), _f(ds.get('RescaleIntercept', 0), 0.0)
    else:
        per, sh = ds.get('PerFrameFunctionalGroupsSequence'), _seq0(ds, 'SharedFunctionalGroupsSequence')
        g = per[k] if per is not None and k < len(per) else None

        def grp(name):
            for src in (g, sh):
                v = _seq0(src, name) if src is not None else None
                if v is not None:
                    return v
            return None
        pp, po = grp('PlanePositionSequence'), grp('PlaneOrientationSequence')
        pm, pv = grp('PixelMeasuresSequence'), grp('PixelValueTransformationSequence')
        ipp = pp.get('ImagePositionPatient') if pp is not None else None
        iop = po.get('ImageOrientationPatient') if po is not None else ds.get('ImageOrientationPatient')
        ps = pm.get('PixelSpacing') if pm is not None else ds.get('PixelSpacing')
        src = pv if pv is not None else ds
        slope, inter = _f(src.get('RescaleSlope', 1), 1.0), _f(src.get('RescaleIntercept', 0), 0.0)
    if ipp is None or iop is None or ps is None:
        return None
    try:
        ipp, iop, ps = (np.array([float(v) for v in a]) for a in (ipp, iop, ps))
    except Exception:
        return None
    if ipp.shape != (3,) or iop.shape != (6,) or ps.shape != (2,) or ps.min() <= 0:
        return None
    return ipp, iop, ps, (slope or 1.0), (inter or 0.0)


def load_series(d):
    files = sorted(f for f in os.listdir(d) if f.lower().endswith('.dcm'))
    if not files:
        raise FileNotFoundError(f'no .dcm files in {d}')
    frames, tsx = [], Counter()
    info = dict(files=len(files), fallback=0, multiframe=0, skipped=0, mono1=0)
    for f in files:
        p = os.path.join(d, f)
        try:
            ds = pydicom.dcmread(p)
        except Exception:
            info['skipped'] += 1
            continue
        try:
            tsx[str(ds.file_meta.TransferSyntaxUID.name)] += 1
        except Exception:
            pass
        try:
            a, rescaled = _decode(ds, p)
        except Exception:
            info['skipped'] += 1
            continue
        a = np.asarray(a)
        if a.ndim == 2:
            a = a[None]
        if a.ndim != 3:
            info['skipped'] += 1
            continue
        info['fallback'] += int(rescaled)
        info['multiframe'] += int(a.shape[0] > 1)
        mono1 = str(ds.get('PhotometricInterpretation', '')).upper() == 'MONOCHROME1'
        echo, inst = _f(ds.get('EchoTime', None), 0.0) or 0.0, _f(ds.get('InstanceNumber', None), 0.0) or 0.0
        thk = _f(ds.get('SliceThickness', None), None)
        for k in range(a.shape[0]):
            g = _frame_geom(ds, k, a.shape[0])
            if g is None:
                info['skipped'] += 1
                continue
            ipp, iop, ps, slope, inter = g
            if rescaled:
                slope, inter = 1.0, 0.0
            frames.append(dict(px=a[k], ipp=ipp, iop=iop, ps=ps, slope=slope, inter=inter, mono1=mono1, echo=echo,
                               inst=inst * 10000 + k, thk=thk))
    if not frames:
        raise ValueError('no readable image in the series')

    def key(x):
        return x['px'].shape, tuple(np.round(x['iop'], 2))
    best, count = Counter(key(x) for x in frames).most_common(1)[0]
    if count < MIN_SERIES_IMAGES:
        raise ValueError(f'only {count} images with one geometry (localizer?)')
    frames = [x for x in frames if key(x) == best]
    iop = frames[0]['iop']
    r, c = iop[:3] / np.linalg.norm(iop[:3]), iop[3:] / np.linalg.norm(iop[3:])
    n = np.cross(r, c)
    n = n / np.linalg.norm(n)
    pos = np.round(np.array([x['ipp'] @ n for x in frames]), 1)
    if len(np.unique(pos)) < len(pos):
        E = {}
        for x, q in zip(frames, pos):
            E.setdefault(x['echo'], set()).add(q)
        if len(E) > 1:
            e0 = sorted(E, key=lambda e: (-len(E[e]), e))[0]
            keep = [i for i, x in enumerate(frames) if x['echo'] == e0]
            frames, pos = [frames[i] for i in keep], pos[keep]
        seen, keep = set(), []
        for i in sorted(range(len(frames)), key=lambda i: (pos[i], frames[i]['inst'])):
            if pos[i] not in seen:
                seen.add(pos[i])
                keep.append(i)
        info['duplicates_dropped'] = len(frames) - len(keep)
        frames = [frames[i] for i in keep]
    w = np.array([x['ipp'] @ n for x in frames])
    o = np.argsort(w, kind='stable')
    frames, w = [frames[i] for i in o], w[o]
    vol = np.stack([x['px'].astype(np.float32) * np.float32(x['slope']) + np.float32(x['inter']) for x in frames])
    if sum(x['mono1'] for x in frames) * 2 > len(frames):
        vol = vol.max() - vol
    info.update(used=len(frames), tsx='|'.join(sorted(tsx)), shape=list(vol.shape))
    return SimpleNamespace(vol=vol, ipp=np.array([x['ipp'] for x in frames]), r=r, c=c, n=n, w=w,
                           dy=float(frames[0]['ps'][0]), dx=float(frames[0]['ps'][1]),
                           dz=float(np.median(np.diff(w))) if len(w) > 1 else float(frames[0]['thk'] or 3.0),
                           info=info, uid=os.path.basename(os.path.normpath(d)))


def load_mask(path):
    import SimpleITK as sitk
    m = sitk.ReadImage(path)
    return dict(arr=sitk.GetArrayFromImage(m).astype(np.uint8), D=np.array(m.GetDirection(), float).reshape(3, 3),
                o=np.array(m.GetOrigin(), float), sp=np.array(m.GetSpacing(), float))


def mask_at(M, pts):
    q = ((pts - M['o']) @ M['D']) / M['sp']
    idx = np.rint(q).astype(np.int64)
    size = np.array(M['arr'].shape[::-1])
    ok = np.all((idx >= 0) & (idx < size), axis=-1)
    out = np.zeros(pts.shape[:-1], np.uint8)
    ii = idx[ok]
    out[ok] = M['arr'][ii[:, 2], ii[:, 1], ii[:, 0]]
    return out


class PlaneError(ValueError):
    pass


def canon_axes(view, s):
    med, inf, post = np.array([-s, 0.0, 0.0]), np.array([0.0, 0.0, -1.0]), np.array([0.0, 1.0, 0.0])
    return {'sag': (inf, post, med), 'cor': (inf, med, post), 'ax': (post, med, inf)}[view]


def frame_for(V, view, s):
    rt, ct, tt = canon_axes(view, s)
    cos = float(V.n @ tt)
    if abs(cos) < 0.7:
        raise PlaneError(f'series plane is not {view} (normal . expected = {cos:.2f})')
    nc = V.n * np.sign(cos)
    er = rt - (rt @ V.n) * V.n
    er = er / np.linalg.norm(er)
    ec = ct - (ct @ V.n) * V.n
    ec = ec - (ec @ er) * er
    ec = ec / np.linalg.norm(ec)
    return er, ec, nc


def corners(b):
    return np.array([[x, y, z] for x in b[0:2] for y in b[2:4] for z in b[4:6]], float)


def _sample(V, k, C, er, ec, g, sig):
    P = V.ipp[k]
    base = (C - ((C - P) @ V.n) * V.n) - P
    jj = (base @ V.r + g[:, None] * (er @ V.r) + g[None, :] * (ec @ V.r)) / V.dx
    ii = (base @ V.c + g[:, None] * (er @ V.c) + g[None, :] * (ec @ V.c)) / V.dy
    img = V.vol[k]
    H, W = img.shape
    inside = (ii >= -0.5) & (ii <= H - 0.5) & (jj >= -0.5) & (jj <= W - 0.5)
    if not inside.any():
        return np.full(ii.shape, np.nan, np.float32)
    if sig is not None:
        pad = int(4 * max(sig)) + 2
        i0, i1 = max(int(np.floor(ii[inside].min())) - pad, 0), min(int(np.ceil(ii[inside].max())) + pad + 1, H)
        j0, j1 = max(int(np.floor(jj[inside].min())) - pad, 0), min(int(np.ceil(jj[inside].max())) + pad + 1, W)
        sub = ndi.gaussian_filter(img[i0:i1, j0:j1], sig, mode='nearest')
        v = ndi.map_coordinates(sub, [np.clip(ii - i0, 0, i1 - i0 - 1), np.clip(jj - j0, 0, j1 - j0 - 1)], order=1, mode='nearest')
    else:
        v = ndi.map_coordinates(img, [np.clip(ii, 0, H - 1), np.clip(jj, 0, W - 1)], order=1, mode='nearest')
    return np.where(inside, v, np.nan).astype(np.float32)


def crop(V, box, view, s, M=None, out=OUT_PX, margin=MARGIN_MM, max_slices=None, aa=True):
    max_slices = max_slices or MAX_SLICES
    er, ec, nc = frame_for(V, view, s)
    Q = corners(box)
    C = Q.mean(0)
    half = max(np.abs((Q - C) @ er).max(), np.abs((Q - C) @ ec).max()) + margin
    d = 2.0 * half / out
    g = (np.arange(out) - (out - 1) / 2.0) * d
    t = Q @ nc
    t0, t1, tc = t.min() - margin, t.max() + margin, float(C @ nc)
    wc = V.ipp @ nc
    order = np.argsort(wc, kind='stable')
    ws = wc[order]
    cover = float(np.clip((min(t1, ws[-1]) - max(t0, ws[0])) / (t1 - t0), 0, 1))
    n_in = int(((ws >= t0) & (ws <= t1)).sum())
    n_full = max(n_in, min(MIN_SLICES, len(wc))) if V.dz >= THIN_MM else max(1, int(round((t1 - t0) / SLAB_MM)))
    sig = None
    if aa:
        fi, fj = d / V.dy, d / V.dx
        si, sj = (0.5 * np.sqrt(f * f - 1) if f > 1.2 else 0.0 for f in (fi, fj))
        if si > 0 or sj > 0:
            sig = (si, sj)
    if V.dz >= THIN_MM:
        sel = order[(ws >= t0) & (ws <= t1)]
        if len(sel) < MIN_SLICES:
            dist = np.maximum(np.maximum(t0 - wc, wc - t1), 0)
            sel = np.argsort(dist, kind='stable')[:min(MIN_SLICES, len(wc))]
            sel = sel[np.argsort(wc[sel], kind='stable')]
        if len(sel) > max_slices:
            sel = sel[np.round(np.linspace(0, len(sel) - 1, max_slices)).astype(int)]
        groups, centers, mode = [[int(k)] for k in sel], wc[sel], '2d'
    else:
        K = max(1, int(round((t1 - t0) / SLAB_MM)))
        K = min(K, max_slices)
        step = (t1 - t0) / K
        centers, groups, mode = t0 + step * (np.arange(K) + 0.5), [], 'slab'
        for cp in centers:
            mem = np.where(np.abs(wc - cp) <= step / 2)[0]
            if not len(mem):
                k = int(np.argmin(np.abs(wc - cp)))
                mem = np.array([k]) if abs(wc[k] - cp) <= step else np.array([], int)
            groups.append([int(k) for k in mem])
        keep = [i for i, gr in enumerate(groups) if gr]
        if not keep:
            k = int(np.argmin(np.abs(wc - tc)))
            groups, centers, keep = [[k]], np.array([wc[k]]), [0]
        groups, centers = [groups[i] for i in keep], np.asarray(centers)[keep]
    imgs, msks = [], []
    for grp, cp in zip(groups, centers):
        if grp:
            st = np.stack([_sample(V, k, C, er, ec, g, sig) for k in grp])
            with warnings.catch_warnings():
                warnings.simplefilter('ignore', category=RuntimeWarning)
                imgs.append(np.nanmean(st, axis=0) if len(grp) > 1 else st[0])
        else:
            imgs.append(np.full((out, out), np.nan, np.float32))
        if M is not None:
            pts = (C + (cp - tc) * nc) + g[:, None, None] * er + g[None, :, None] * ec
            msks.append(mask_at(M, pts))
    img = np.stack(imgs).astype(np.float32)
    pos = (np.asarray(centers, float) - tc).astype(np.float32)
    meta = dict(d_mm=round(float(d), 4), slice_mm=round(float(np.median(np.diff(centers))) if len(centers) > 1 else float(V.dz), 3),
                mode=mode, n=int(len(centers)), n_in_range=n_in, n_full=int(n_full), cover=round(cover, 3),
                fov=round(float(np.isfinite(img).mean()), 3), aa=None if sig is None else [round(float(x), 2) for x in sig])
    return img, (np.stack(msks) if M is not None else None), pos, meta


def norm_stats(V, box, view, s):
    v = np.array([])
    if box is not None:
        try:
            img = crop(V, box, view, s, None, out=64, margin=0.0, max_slices=16, aa=False)[0]
            v = img[np.isfinite(img)]
        except Exception:
            v = np.array([])
    src = 'bones_box'
    if v.size < 500:
        v, src = V.vol[:, ::4, ::4].ravel(), 'whole_series'
    lo, hi = np.percentile(v, [0.5, 99.5])
    if hi - lo < 1e-3:
        hi = lo + 1.0
    fg = v[v > lo + 0.05 * (hi - lo)]
    p50 = float(np.median(fg)) if fg.size else float(hi)
    return float(lo), float(hi), src, p50


def to_u8(img, lo, hi):
    x = np.clip((img - lo) / (hi - lo), 0.0, 1.0)
    return np.where(np.isfinite(x), x * 255.0 + 0.5, 0).astype(np.uint8)


def enc_stack(u8):
    buf = io.BytesIO()
    im = Image.fromarray(np.ascontiguousarray(u8.reshape(-1, u8.shape[2])), 'L')
    if WEBP:
        im.save(buf, 'WEBP', lossless=True, quality=10, method=1)
    else:
        im.save(buf, 'PNG', compress_level=6)
    return np.frombuffer(buf.getvalue(), np.uint8)


def dec_stack(b, n):
    a = cv2.imdecode(np.frombuffer(np.asarray(b, np.uint8).tobytes(), np.uint8), cv2.IMREAD_GRAYSCALE)
    if a is None:
        raise ValueError('could not decode a crop image')
    return a.reshape(n, -1, a.shape[1])


def decode(z, meta=None):
    keys = list(z.keys()) if hasattr(z, 'keys') else list(z.files)
    if meta is None:
        meta = json.loads(str(z['meta']))
    A = {}
    for name, cm in meta['crops'].items():
        n = int(cm['n'])
        if f'{name}__imgw' in keys:
            A[f'{name}__img'] = dec_stack(z[f'{name}__imgw'], n)
        elif f'{name}__img' in keys:
            A[f'{name}__img'] = np.asarray(z[f'{name}__img'])
        if f'{name}__mskz' in keys:
            A[f'{name}__msk'] = np.frombuffer(zlib.decompress(np.asarray(z[f'{name}__mskz'], np.uint8).tobytes()),
                                              np.uint8).reshape(n, OUT_PX, OUT_PX)
        elif f'{name}__msk' in keys:
            A[f'{name}__msk'] = np.asarray(z[f'{name}__msk'])
        if f'{name}__pos' in keys:
            A[f'{name}__pos'] = np.asarray(z[f'{name}__pos'])
    return A, meta


def read_study(path):
    with np.load(path) as z:
        return decode(z)


def _centroid(m, lab, min_px=150):
    z = np.argwhere(m == lab)
    return z.mean(0) if len(z) >= min_px else None


def orient_checks(msk, view, roi):
    c = {b: _centroid(msk, lab) for lab, b in ((1, 'fem'), (2, 'pat'), (3, 'fib'), (4, 'tib'))}
    out = {}
    if view in ('sag', 'cor') and c['fem'] is not None and c['tib'] is not None:
        out['femur_above_tibia'] = bool(c['fem'][1] < c['tib'][1])
    if c['pat'] is not None and c['fem'] is not None:
        if view == 'sag':
            out['patella_anterior'] = bool(c['pat'][2] < c['fem'][2])
        if view == 'ax':
            out['patella_anterior'] = bool(c['pat'][1] < c['fem'][1])
    if c['fib'] is not None and c['tib'] is not None:
        if view in ('cor', 'ax'):
            out['fibula_lateral'] = bool(c['fib'][2] < c['tib'][2])
        if view == 'sag':
            out['fibula_lateral'] = bool(c['fib'][0] < c['tib'][0])
    if roi == 'medial_TF':
        out['no_fibula_medial'] = bool((msk == 3).sum() < 150)
    return out


def align_shift(u8, msk, d, valid=None, max_mm=8.0):
    b = msk > 0
    ok = [k for k in range(len(b)) if b[k].sum() >= 200]
    if len(ok) < 2:
        return None
    R = max(1, int(round(max_mm / d)))
    H, W = u8.shape[1:]
    N = (H + 2 * R, W + 2 * R)
    frame = np.zeros((H, W), bool)
    frame[2:-2, 2:-2] = True
    acc, nb = 0, 0
    for k in ok:
        im = ndi.gaussian_filter(u8[k].astype(np.float32), 1.0)
        G = np.hypot(ndi.sobel(im, 0), ndi.sobel(im, 1))
        if valid is not None:
            G[~ndi.binary_erosion(valid[k], iterations=3)] = 0
        B = (b[k] & ~ndi.binary_erosion(b[k]) & frame).astype(np.float32)
        acc = acc + np.fft.rfft2(G, N) * np.conj(np.fft.rfft2(B, N))
        nb += B.sum()
    cc = np.fft.irfft2(acc, N) / max(nb, 1.0)
    sc = cc[np.ix_(np.arange(-R, R + 1) % N[0], np.arange(-R, R + 1) % N[1])]
    i, j = np.unravel_index(int(np.argmax(sc)), sc.shape)
    s0 = sc[R, R]
    return dict(dr_mm=round(float((i - R) * d), 2), dc_mm=round(float((j - R) * d), 2),
                gain=round(float(sc[i, j] / s0), 4) if s0 > 0 else None)


def _seq_info(series_rows, uid):
    try:
        r = series_rows[series_rows.SeriesInstanceUID.astype(str) == uid].iloc[0]
    except Exception:
        return {}

    def g(k):
        return r[k] if k in r.index else None
    tr, te = _f(g('TR')), _f(g('TE'))
    txt = f"{g('image_type') or ''} {g('series_desc') or ''}".upper()
    seq = str(g('scanning_sequence') or '').upper()
    return dict(desc=str(g('series_desc'))[:40], tr=tr, te=te,
                gre=bool(('GR' in seq and 'SE' not in seq) or any(w in txt for w in GRE_WORDS)),
                kind=None if tr is None or tr != tr else ('T1' if tr < 1000 else 'PD' if (te is not None and te <= 60) else 'T2'))


def study_crops(sid, series_rows, boxes, lat, series_dir, mask_path=None, ref_uid=None, box_src=None, crops=CROPS):
    s = 1.0 if lat == 'L' else -1.0
    M = load_mask(mask_path) if mask_path and os.path.exists(mask_path) else None
    ranked = rank_series(series_rows)
    A, used = {}, set()
    meta = dict(StudyInstanceUID=sid, version=VERSION, lat=lat, side_sign=s, mask=M is not None, slots={}, crops={}, errors=[])
    for slot in SLOTS:
        need = [r for r, sl in crops if sl == slot and boxes.get(r) is not None]
        if not need:
            continue
        labelled = [u for u in ranked.get(slot, [])[:3] if u not in used]
        others = [u for sl2, lst in ranked.items()
                  if sl2[-3:] == slot[-3:] and VIEW[sl2] != VIEW[slot]
                  for u in lst if u not in used and u not in labelled][:2]
        cands = labelled + others
        if not cands:
            meta['slots'][slot] = dict(missing='no series of this kind')
            continue
        V, tried = None, []
        for uid in cands:
            try:
                V = load_series(os.path.join(series_dir, uid))
                frame_for(V, VIEW[slot], s)
                break
            except Exception as e:
                tried.append(f'{uid[-10:]}: {type(e).__name__}: {str(e)[:100]}')
                V = None
        if V is None:
            if labelled:
                meta['slots'][slot] = dict(missing='no usable series', tried=tried)
                meta['errors'].append(f'{slot}: no usable series ({"; ".join(tried)})')
            else:
                meta['slots'][slot] = dict(missing='no series of this kind')
            continue
        used.add(V.uid)
        lo, hi, nsrc, p50 = norm_stats(V, boxes.get('bones'), VIEW[slot], s)
        sm = dict(uid=V.uid, rank=cands.index(V.uid), relabelled=V.uid not in ranked.get(slot, []), lo=round(lo, 3),
                  hi=round(hi, 3), p50=round(p50, 3), norm=nsrc, dz=round(V.dz, 3), dy=round(V.dy, 4), dx=round(V.dx, 4),
                  is_ref=bool(ref_uid is not None and V.uid == ref_uid), info=V.info, failed_before=tried,
                  seq=_seq_info(series_rows, V.uid))
        for roi in need:
            name = f'{roi}__{slot}'
            try:
                img, msk, pos, cm = crop(V, boxes[roi], VIEW[slot], s, M, max_slices=MAX_LARGE if roi in LARGE_ROIS else MAX_SLICES)
                u8 = to_u8(img, lo, hi)
                A[f'{name}__imgw'], A[f'{name}__pos'] = enc_stack(u8), pos.astype(np.float16)
                if msk is not None:
                    A[f'{name}__mskz'] = np.frombuffer(zlib.compress(np.ascontiguousarray(msk).tobytes(), 6), np.uint8)
                    cm['checks'] = orient_checks(msk, VIEW[slot], roi)
                    cm['bone_px'] = {b: int((msk == lab).sum()) for lab, b in ((1, 'femur'), (2, 'patella'), (3, 'fibula'), (4, 'tibia'))}
                    if roi == ANCHOR[slot]:
                        sm['align'] = align_shift(u8, msk, cm['d_mm'], np.isfinite(img))
                cm.update(series=V.uid, slot=slot, roi=roi, box_source=(box_src or {}).get(roi))
                meta['crops'][name] = cm
            except Exception as e:
                meta['errors'].append(f'{name}: {type(e).__name__}: {str(e)[:150]}')
        meta['slots'][slot] = sm
        del V
    return A, meta


def js(o):
    return o.item() if hasattr(o, 'item') else str(o)

In [ ]:
%%writefile knee/cropping.py
import os
import json
import time
import argparse
from concurrent.futures import ProcessPoolExecutor
from concurrent.futures.process import BrokenProcessPool
import numpy as np
import pandas as pd
from knee import crops

S = {}


def setup(work, strict):
    SM = pd.read_csv(f'{work}/meta/series_meta.csv')
    RO = pd.read_csv(f'{work}/meta/rois.csv')
    QF = pd.read_csv(f'{work}/meta/regions.csv', usecols=lambda c: c in ('StudyInstanceUID', 'SeriesInstanceUID', 'site_key'))
    ST = pd.read_csv(f'{work}/meta/studies.csv', dtype=str)
    S.update(work=work, strict=strict, empty=SM.iloc[:0], series={k: g for k, g in SM.groupby('StudyInstanceUID')},
             boxes={sid: {r.roi: [r.xmin, r.xmax, r.ymin, r.ymax, r.zmin, r.zmax] for r in g.itertuples()}
                    for sid, g in RO.groupby('StudyInstanceUID')},
             sources={sid: dict(zip(g.roi, g.source)) for sid, g in RO.groupby('StudyInstanceUID')},
             side=RO.groupby('StudyInstanceUID').lat.first().to_dict(), ref=dict(zip(QF.StudyInstanceUID, QF.SeriesInstanceUID)),
             site=dict(zip(QF.StudyInstanceUID, QF.site_key)) if 'site_key' in QF else {},
             dirs=dict(zip(ST.StudyInstanceUID, ST.study_dir)))


def study(sid):
    W, t0 = S['work'], time.time()
    out = f'{W}/crops/{sid}.npz'
    if os.path.exists(out):
        return dict(StudyInstanceUID=sid, skipped=1)
    try:
        if sid not in S['boxes']:
            raise KeyError('no region boxes')
        lat = S['side'].get(sid)
        lat = lat if lat in ('L', 'R') else None
        Ar, meta = crops.study_crops(sid, S['series'].get(sid, S['empty']), S['boxes'][sid], lat, S['dirs'][sid],
                                     f'{W}/masks/{sid}.nii.gz', S['ref'].get(sid), S['sources'].get(sid))
        if S['strict'] and meta['errors']:
            raise RuntimeError(f'crop errors {meta["errors"][:3]}')
        meta['site_key'] = S['site'].get(sid)
        Ar['meta'] = np.array(json.dumps(meta, default=crops.js))
        tmp = f'{W}/crops/{sid}.tmp.npz'
        np.savez(tmp, **Ar)
        os.replace(tmp, out)
        return dict(StudyInstanceUID=sid, n_crops=len(meta['crops']), problems=len(meta['errors']), sec=round(time.time() - t0, 2))
    except Exception as e:
        if S['strict']:
            raise RuntimeError(f'{sid}: {e!r}') from e
        return dict(StudyInstanceUID=sid, fatal=f'{type(e).__name__}: {str(e)[:300]}', sec=round(time.time() - t0, 2))


def run_all(todo, nproc, deadline, work, strict):
    handled, remaining = set(), list(todo)
    while remaining:
        try:
            with ProcessPoolExecutor(nproc, initializer=setup, initargs=(work, strict)) as pool:
                for m in pool.map(study, remaining, chunksize=1):
                    handled.add(m['StudyInstanceUID'])
                    yield m
                    if time.time() > deadline:
                        print('[crops] time limit', flush=True)
                        pool.shutdown(wait=False, cancel_futures=True)
                        return
            remaining = []
        except BrokenProcessPool:
            if strict:
                raise
            remaining = [s for s in remaining if s not in handled]
            for s in remaining[:nproc]:
                try:
                    with ProcessPoolExecutor(1, initializer=setup, initargs=(work, strict)) as p1:
                        m = p1.submit(study, s).result()
                except BrokenProcessPool:
                    m = dict(StudyInstanceUID=s, fatal='worker died')
                handled.add(s)
                yield m
            remaining = remaining[nproc:]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--work', required=True)
    ap.add_argument('--nproc', type=int, default=4)
    ap.add_argument('--deadline', type=float, default=1e18)
    ap.add_argument('--strict', type=int, default=1)
    A = ap.parse_args()
    os.makedirs(f'{A.work}/crops', exist_ok=True)
    todo = pd.read_csv(f'{A.work}/meta/studies.csv', dtype=str).StudyInstanceUID.tolist()
    print(f'[crops] {len(todo)} studies', flush=True)
    t0, n, bad, prob = time.time(), 0, 0, 0
    for m in run_all(todo, A.nproc, A.deadline, A.work, bool(A.strict)):
        n += 1
        bad += int('fatal' in m)
        prob += int(m.get('problems', 0) > 0)
        if n % 100 == 0 or n == len(todo):
            el = time.time() - t0
            print(f'[crops] {n}/{len(todo)} | failed {bad} | {el / n:.2f} s/study | {el * (len(todo) - n) / n / 60:.0f} min left',
                  flush=True)
    print(f'[crops] done: {n} studies in {(time.time() - t0) / 60:.1f} min | failed {bad} | with a missing crop {prob}', flush=True)


if __name__ == '__main__':
    main()

In [ ]:
%%writefile knee/data.py
import math
import numpy as np
import torch
import torch.nn.functional as F
from . import crops


def slice_index(S, K, train, rng):
    if S <= K:
        return np.arange(S)
    if not train:
        return np.unique(np.round(np.linspace(0, S - 1, K)).astype(int))
    edges = np.linspace(0, S, K + 1)
    return np.array([rng.integers(int(math.floor(a)), max(int(math.floor(a)) + 1, int(math.ceil(b))))
                     for a, b in zip(edges[:-1], edges[1:])]).clip(0, S - 1)


class StudyData:

    def __init__(self, crops_dir, crop_names, K_train=8, K_eval=12, min_fov=0.25):
        self.dir, self.names = crops_dir, list(crop_names)
        self.K_train, self.K_eval, self.min_fov = K_train, K_eval, min_fov

    def load(self, sid, train, rng):
        A, meta = crops.read_study(f'{self.dir}/{sid}.npz')
        K = self.K_train if train else self.K_eval
        H = crops.OUT_PX
        x = np.zeros((len(self.names), K, 3, H, H), np.uint8)
        valid = np.zeros((len(self.names), K), bool)
        pos = np.zeros((len(self.names), K), np.float32)
        fov = np.zeros(len(self.names), np.float32)
        for j, n in enumerate(self.names):
            if f'{n}__img' not in A:
                continue
            fov[j] = float(meta['crops'].get(n, {}).get('fov', 1.0))
            if fov[j] < self.min_fov:
                continue
            img, p = A[f'{n}__img'], A[f'{n}__pos'].astype(np.float32)
            S = len(img)
            idx = slice_index(S, K, train, rng)
            for t, i in enumerate(idx):
                x[j, t, 0], x[j, t, 1], x[j, t, 2] = img[max(i - 1, 0)], img[i], img[min(i + 1, S - 1)]
            valid[j, :len(idx)] = True
            pos[j, :len(idx)] = p[idx] / 50.0
        return x, valid, pos, fov


def augment_gpu(x, valid, gen=None, rot_deg=10.0, scale=0.1, shift=0.06, contrast=0.15, bright=0.08, gamma=0.15, drop_crop=0.05):
    B, Cn, K = valid.shape
    dev = x.device

    def r(*s):
        return torch.rand(*s, device=dev, generator=gen)
    ang = (r(B, Cn) * 2 - 1) * math.radians(rot_deg)
    sc = 1 + (r(B, Cn) * 2 - 1) * scale
    tx, ty = (r(B, Cn) * 2 - 1) * shift * 2, (r(B, Cn) * 2 - 1) * shift * 2
    th = torch.stack([torch.stack([torch.cos(ang) / sc, -torch.sin(ang) / sc, tx], -1),
                      torch.stack([torch.sin(ang) / sc, torch.cos(ang) / sc, ty], -1)], -2)
    m = valid.reshape(-1)
    flat = x.reshape(B * Cn * K, *x.shape[3:])[m]
    thf = th[:, :, None].expand(B, Cn, K, 2, 3).reshape(-1, 2, 3)[m]
    grid = F.affine_grid(thf, flat.shape, align_corners=False)
    flat = F.grid_sample(flat, grid, mode='bilinear', padding_mode='zeros', align_corners=False)
    c = (1 + (r(B, Cn) * 2 - 1) * contrast)[:, :, None].expand(B, Cn, K).reshape(-1)[m]
    b = ((r(B, Cn) * 2 - 1) * bright)[:, :, None].expand(B, Cn, K).reshape(-1)[m]
    g = (1 + (r(B, Cn) * 2 - 1) * gamma)[:, :, None].expand(B, Cn, K).reshape(-1)[m]
    flat = (flat.clamp(0, 1) ** g[:, None, None, None] * c[:, None, None, None] + b[:, None, None, None]).clamp(0, 1)
    out = torch.zeros_like(x.reshape(B * Cn * K, *x.shape[3:]))
    out[m] = flat
    keep = r(B, Cn) >= drop_crop
    has = (valid.any(-1) & keep).any(-1, keepdim=True)
    keep = torch.where(has, keep, torch.ones_like(keep))
    return out.reshape(x.shape), valid & keep[:, :, None]

In [ ]:
%%writefile knee/model.py
import math
import numpy as np
import torch
import torch.nn as nn
import timm

PRIOR_BIAS = {0: 0.0, 1: -1.0, 2: -3.0}


class KneeNet(nn.Module):

    def __init__(self, backbone, n_crops, relevance, d=256, n_layers=1, heads=4, drop=0.1):
        super().__init__()
        self.enc = timm.create_model(backbone, pretrained=False, num_classes=0, in_chans=3)
        nf = self.enc.num_features
        self.proj = nn.Sequential(nn.Linear(nf, d), nn.LayerNorm(d))
        self.crop_emb = nn.Embedding(n_crops, d)
        self.pos = nn.Sequential(nn.Linear(1, d // 4), nn.GELU(), nn.Linear(d // 4, d))
        layer = nn.TransformerEncoderLayer(d, heads, 2 * d, dropout=drop, batch_first=True, norm_first=True)
        self.mix = nn.TransformerEncoder(layer, n_layers, enable_nested_tensor=False) if n_layers else None
        R = np.asarray(relevance)
        L = R.shape[0]
        self.q = nn.Parameter(torch.randn(L, d) * 0.02)
        self.bias = nn.Parameter(torch.tensor(np.vectorize(PRIOR_BIAS.get)(R), dtype=torch.float32))
        self.drop = nn.Dropout(0.2)
        self.head_w = nn.Parameter(torch.randn(L, d) * 0.02)
        self.head_b = nn.Parameter(torch.zeros(L))
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
        self.d = d

    def forward(self, x, valid, pos):
        B, Cn, K = valid.shape
        flat = x.reshape(B * Cn * K, *x.shape[3:])[valid.reshape(-1)]
        f = self.proj(self.enc((flat - self.mean) / self.std).float())
        tok = torch.zeros(B * Cn * K, self.d, device=x.device, dtype=f.dtype)
        tok[valid.reshape(-1)] = f
        tok = tok.view(B, Cn, K, self.d) + self.crop_emb.weight[None, :, None, :] + self.pos(pos[..., None].float())
        tok = tok.view(B, Cn * K, self.d)
        m = valid.view(B, Cn * K)
        if self.mix is not None:
            tok = self.mix(tok, src_key_padding_mask=~m)
        crop_of = torch.arange(Cn, device=x.device).repeat_interleave(K)
        att = torch.einsum('bnd,ld->bln', tok, self.q) / math.sqrt(self.d) + self.bias[:, crop_of][None]
        att = att.masked_fill(~m[:, None, :], -10000.0).softmax(-1)
        pooled = self.drop(torch.einsum('bln,bnd->bld', att, tok))
        return (pooled * self.head_w).sum(-1) + self.head_b, att


def build(cfg, relevance):
    return KneeNet(cfg['backbone'], cfg['n_crops'], relevance, d=cfg.get('d', 256), n_layers=cfg.get('n_layers', 1),
                   heads=cfg.get('heads', 4))

In [ ]:
%%writefile knee/scoring.py
import json
import time
import math
import argparse
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from knee import crops
from knee.findings import LABELS, relevance
from knee.data import slice_index
from knee.model import build

PASSES = [(0.0, 1.0), (5.0, 1.0), (-5.0, 1.0), (0.0, 1.05)]


class Studies(Dataset):

    def __init__(self, files, rows, names, min_fov, strict):
        self.files, self.rows, self.names, self.min_fov, self.strict = files, rows, names, min_fov, strict

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = int(self.rows[i])
        f = self.files[r]
        try:
            A, meta = crops.read_study(f)
        except Exception as e:
            if self.strict:
                raise RuntimeError(f'crops of {f}: {e!r}') from e
            return None, None, None, r, f'{type(e).__name__}: {str(e)[:150]}'
        S = np.zeros(len(self.names), np.int64)
        imgs, poss = {}, {}
        for j, n in enumerate(self.names):
            if f'{n}__img' not in A:
                continue
            if float(meta['crops'].get(n, {}).get('fov', 1.0)) < self.min_fov:
                continue
            imgs[j], poss[j] = A[f'{n}__img'], A[f'{n}__pos'].astype(np.float32)
            S[j] = len(imgs[j])
        H = crops.OUT_PX
        IMG = np.zeros((len(self.names), max(1, int(S.max())), H, H), np.uint8)
        POS = np.zeros(IMG.shape[:2], np.float32)
        for j in imgs:
            IMG[j, :S[j]] = imgs[j]
            POS[j, :S[j]] = poss[j]
        return torch.from_numpy(IMG), torch.from_numpy(POS), torch.from_numpy(S), r, ''


def autocast(amp):
    return torch.autocast(device_type='cuda', dtype=torch.float16) if amp else torch.autocast(device_type='cpu', enabled=False)


def model_input(IMG, POS, S_t, S_np, dev, K):
    idx = np.zeros((len(S_np), K), np.int64)
    val = np.zeros(idx.shape, bool)
    for j, s in enumerate(S_np):
        if s > 0:
            x = slice_index(int(s), K, False, None)
            idx[j, :len(x)] = x
            val[j, :len(x)] = True
    idx_t, val_t = torch.from_numpy(idx).to(dev), torch.from_numpy(val).to(dev)
    C = len(S_np)
    top = (S_t - 1).clamp(min=0)[:, None].expand(C, K)
    lo, hi = (idx_t - 1).clamp(min=0), torch.minimum(idx_t + 1, top)
    c = torch.arange(C, device=dev)[:, None].expand(C, K)
    x = torch.stack([IMG[c, lo], IMG[c, idx_t], IMG[c, hi]], 2).float().div_(255)
    x.mul_(val_t[..., None, None, None].to(x.dtype))
    pos = POS[c, idx_t] / 50.0 * val_t
    xs = []
    for rot, zoom in PASSES:
        if rot == 0.0 and zoom == 1.0:
            xs.append(x)
            continue
        t = math.radians(rot)
        th = torch.tensor([[math.cos(t) / zoom, -math.sin(t) / zoom, 0.0], [math.sin(t) / zoom, math.cos(t) / zoom, 0.0]],
                          device=dev, dtype=x.dtype)
        flat, m = x.reshape(-1, *x.shape[2:]), val_t.reshape(-1)
        out = torch.zeros_like(flat)
        sub = flat[m]
        if len(sub):
            grid = F.affine_grid(th[None].expand(len(sub), 2, 3), list(sub.shape), align_corners=False)
            out[m] = F.grid_sample(sub, grid, mode='bilinear', padding_mode='zeros', align_corners=False).clamp(0, 1)
        xs.append(out.reshape(x.shape))
    B = len(PASSES)
    return torch.stack(xs), val_t[None].expand(B, C, K).contiguous(), pos[None].expand(B, C, K).contiguous()


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--job', required=True)
    ap.add_argument('--part', type=int, default=0)
    ap.add_argument('--parts', type=int, default=1)
    ap.add_argument('--deadline', type=float, default=1e18)
    ap.add_argument('--strict', type=int, default=1)
    a = ap.parse_args()
    J = json.load(open(a.job))
    dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    amp = dev.type == 'cuda'
    T = pd.read_csv(J['studies'], dtype=str)
    rows = np.arange(len(T))[a.part::a.parts]
    names = list(J['crop_names'])
    R = relevance(names)
    models = {}
    for fam in J['families']:
        cfg = json.load(open(fam['config']))
        models[fam['name']] = []
        for path in fam['weights']:
            net = build(cfg, R)
            net.load_state_dict(torch.load(path, map_location='cpu')['state_dict'])
            models[fam['name']].append(net.to(dev).eval())
    P = {k: np.full((len(rows), len(LABELS)), np.nan, np.float32) for k in models}
    where = {int(r): i for i, r in enumerate(rows)}
    print(f'[score {a.part}] {len(rows)} studies | ' + ', '.join(f'{k} {len(v)}' for k, v in models.items()), flush=True)
    ds = Studies(T.crops_file.tolist(), rows, names, float(J['min_fov']), bool(a.strict))
    dl = DataLoader(ds, batch_size=None, shuffle=False, num_workers=int(J['workers']), pin_memory=amp,
                    prefetch_factor=4 if int(J['workers']) > 0 else None)
    t0, done, failed = time.time(), 0, 0
    with torch.no_grad():
        for IMG, POS, S, r, err in dl:
            if time.time() > a.deadline:
                print(f'[score {a.part}] time limit after {done} studies', flush=True)
                break
            i = where[r]
            if err:
                failed += 1
            else:
                S_np = S.numpy()
                IMG, POS, S_t = IMG.to(dev, non_blocking=True), POS.to(dev, non_blocking=True), S.to(dev)
                x, val, pos = model_input(IMG, POS, S_t, S_np, dev, int(J['k_eval']))
                for k, nets in models.items():
                    acc = np.zeros(len(LABELS), np.float64)
                    for net in nets:
                        with autocast(amp):
                            lg, _ = net(x, val, pos)
                        acc += torch.sigmoid(lg.float()).mean(0).cpu().numpy()
                    P[k][i] = acc / len(nets)
            done += 1
            if done % 100 == 0 or done == len(rows):
                el = time.time() - t0
                print(f'[score {a.part}] {done}/{len(rows)} | {done / el:.2f} studies/s | {(len(rows) - done) * el / done / 60:.0f} min left',
                      flush=True)
    np.savez(f'{J["out"]}/scores_{a.part}of{a.parts}.npz', rows=rows, **P)
    print(f'[score {a.part}] done: {done} studies ({failed} without crops) in {(time.time() - t0) / 60:.1f} min', flush=True)


if __name__ == '__main__':
    main()

In [ ]:
%%writefile knee/launch.py
import os
import sys
import time
import subprocess

ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))


def worker(module, *args):
    return [sys.executable, '-m', f'knee.{module}', *[str(x) for x in args]]


def env(**extra):
    e = dict(os.environ, PYTHONPATH=ROOT + os.pathsep + os.environ.get('PYTHONPATH', ''))
    e.update({k: str(v) for k, v in extra.items()})
    return e


def _start(cmd, e, log):
    return dict(proc=subprocess.Popen(cmd, stdout=open(log, 'w'), stderr=subprocess.STDOUT, env=e, cwd=ROOT),
                log=log, pos=0, rest='')


def _pump(p):
    with open(p['log'], errors='replace') as fh:
        fh.seek(p['pos'])
        text = p['rest'] + fh.read()
        p['pos'] = fh.tell()
    lines = text.split('\n')
    p['rest'] = lines[-1]
    for line in lines[:-1]:
        if line.startswith('['):
            print(line, flush=True)
    if p['proc'].poll() is None:
        return None
    if p['rest'].startswith('['):
        print(p['rest'], flush=True)
    return p['proc'].returncode


def _stop(running, p):
    for q in running:
        q['proc'].kill()
    tail = open(p['log'], errors='replace').read()[-4000:]
    raise RuntimeError(f'worker stopped with exit code {p["proc"].returncode} ({p["log"]}):\n{tail}')


def run_parallel(jobs, strict=True, poll=15):
    running = [_start(cmd, e, log) for cmd, e, log in jobs]
    failed = []
    while running:
        time.sleep(poll)
        for p in list(running):
            code = _pump(p)
            if code is None:
                continue
            running.remove(p)
            if code != 0:
                if strict:
                    _stop(running, p)
                failed.append(p)
    for p in failed:
        tail = open(p['log'], errors='replace').read()[-600:].replace('\n', ' | ')
        print(f'[worker] exit code {p["proc"].returncode}: {tail}', flush=True)
    return len(failed)


def run_queue(jobs, n_slots, deadline=0, poll=15):
    queue, running = list(jobs), {}
    while queue or running:
        if queue and deadline and time.time() > deadline - 20 * 60:
            print(f'[queue] not started, too close to the time limit: {[os.path.dirname(log) for _, _, log in queue]}', flush=True)
            queue = []
        for slot in range(n_slots):
            if slot not in running and queue:
                cmd, e, log = queue.pop(0)
                running[slot] = _start(cmd, dict(e, CUDA_VISIBLE_DEVICES=str(slot)), log)
        time.sleep(poll)
        for slot, p in list(running.items()):
            code = _pump(p)
            if code is None:
                continue
            del running[slot]
            if code != 0:
                _stop(list(running.values()), p)

In [ ]:
%%writefile predict.py
import os
import json
import time
import glob
import argparse
import numpy as np
import pandas as pd
import torch
from knee import series, crops
from knee.findings import LABELS
from knee.launch import worker, env, run_parallel


def ids_from_csv(path, strict):
    if not os.path.exists(path):
        return None
    C = pd.read_csv(path, dtype=str)
    if 'StudyInstanceUID' not in C and strict:
        raise RuntimeError(f'{path} has no StudyInstanceUID column: {list(C.columns)}')
    return list(dict.fromkeys(C['StudyInstanceUID' if 'StudyInstanceUID' in C else C.columns[0]].dropna().str.strip()))


def find_studies(a, W, t0):
    data = a.data
    test_dir = f'{data}/test_series'
    if not os.path.isdir(test_dir):
        cand = [d for d in series.subdirs(data) if d.lower().startswith('test')]
        if cand:
            test_dir = f'{data}/{cand[0]}'
    folders = series.subdirs(test_dir)
    fset = set(folders)
    s_ids, t_ids = ids_from_csv(f'{data}/sample_submission.csv', False), ids_from_csv(f'{data}/test.csv', False)
    hidden = os.getenv('KAGGLE_IS_COMPETITION_RERUN') is not None or max(len(s_ids or []), len(t_ids or []), len(folders)) > 50
    strict = {'auto': not hidden, 'strict': True, 'lenient': False}[a.mode]
    if strict:
        s_ids, t_ids = ids_from_csv(f'{data}/sample_submission.csv', True), ids_from_csv(f'{data}/test.csv', True)
    fit = [(c, n) for c, n in ((s_ids, 'sample_submission.csv'), (t_ids, 'test.csv'))
           if c and (not fset or len(fset.intersection(c)) >= 0.5 * len(c))]
    test_ids, id_src = max(fit, key=lambda x: len(x[0])) if fit else (folders, 'folders')
    if fset and len(test_ids) < 0.5 * len(fset):
        test_ids, id_src = folders, 'folders'
    if not test_ids:
        raise RuntimeError(f'no test study found in {data}: {sorted(os.listdir(data))}')
    if strict and s_ids is not None and t_ids is not None and set(s_ids) != set(t_ids):
        raise RuntimeError(f'sample_submission.csv and test.csv list different studies ({len(s_ids)} vs {len(t_ids)})')
    no_dir = [s for s in test_ids if s not in fset]
    if no_dir and (strict or len(no_dir) == len(test_ids)):
        raise RuntimeError(f'{len(no_dir)} of {len(test_ids)} test studies have no folder in {test_dir}: {no_dir[:5]}')
    check_ids = []
    if not hidden and a.check_crops and os.path.exists(f'{data}/train_series.csv'):
        TR = pd.read_csv(f'{data}/train_series.csv', dtype={'StudyInstanceUID': str})
        have = [s for s in sorted(TR.StudyInstanceUID.unique()) if os.path.exists(f'{a.check_crops}/{s}.npz')]
        check_ids = [have[i] for i in np.random.default_rng(7).choice(len(have), min(6, len(have)), replace=False)]
    TS = pd.concat([series.series_rows(test_ids, test_dir, f'{data}/test_series.csv', 'test'),
                    series.series_rows(check_ids, f'{data}/train_series', f'{data}/train_series.csv', 'train')], ignore_index=True)
    n_rebuilt = 0
    if not strict:
        TS, n_rebuilt = series.rebuild_folders(TS, test_ids, test_dir, fset, f'{W}/mirror')
    no_img = sorted(set(test_ids) - set(TS[TS.split == 'test'].StudyInstanceUID))
    if no_img and (strict or len(no_img) == len(test_ids)):
        raise RuntimeError(f'{len(no_img)} test studies have no series: {no_img[:5]}')
    SM = series.add_headers(TS, strict)
    SM.drop(columns=['series_dir', 'split', 'plane_hdr', 'fs_hdr']).to_csv(f'{W}/meta/series_meta.csv', index=False)
    studies = series.study_table(SM)
    no_ref = studies[studies.ref_dir == ''].StudyInstanceUID.tolist()
    if no_ref and strict:
        raise RuntimeError(f'{len(no_ref)} studies have no usable sagittal series: {no_ref[:5]}')
    studies.to_csv(f'{W}/meta/studies.csv', index=False)
    print(f'test studies {len(test_ids)} from {id_src} | series {int((SM.split == "test").sum())} | '
          f'header errors {int(SM.error.notna().sum())} | no sagittal {len(no_ref)} | no folder {len(no_dir)} | '
          f'rebuilt folders {n_rebuilt} | check studies {len(check_ids)} | strict {strict} | {(time.time() - t0) / 60:.1f} min', flush=True)
    return dict(test_ids=test_ids, check_ids=check_ids, strict=strict, series=SM, studies=studies, no_ref=no_ref)


def gpu_env(n_gpu, i, **extra):
    return env(CUDA_VISIBLE_DEVICES=i % n_gpu, **extra) if n_gpu else env(**extra)


def run_preprocessing(a, W, t0, limit, run):
    strict, studies, n_all = run['strict'], run['studies'], len(run['studies'])
    n_gpu = torch.cuda.device_count()
    seg_table = f'{W}/meta/seg_table.csv'
    studies[studies.ref_dir != ''][['StudyInstanceUID', 'ref_dir', 'ref_uid', 'alt_dir', 'alt_uid', 'lat_tag', 'site_key']].to_csv(
        seg_table, index=False)
    nw = 2 * max(1, n_gpu)
    deadline = t0 + limit - n_all * 4.5 - 600
    nn = dict(KNEE_THREADS=1, nnUNet_raw=f'{W}/nnunet', nnUNet_preprocessed=f'{W}/nnunet', nnUNet_results=f'{W}/nnunet')
    run_parallel([(worker('segment', '--table', seg_table, '--model', a.segmentation, '--out', f'{W}/seg', '--shard', i,
                          '--nshards', nw, '--deadline', deadline, '--strict', int(strict)),
                   gpu_env(n_gpu, i, **nn), f'{W}/seg/log{i}.txt') for i in range(nw)], strict)
    print(f'segmentation: {len(glob.glob(f"{W}/seg/qc/*.json"))} of {n_all} studies | {(time.time() - t0) / 60:.1f} min', flush=True)
    run_parallel([(worker('regions', '--work', W, '--anatomy', f'{a.models}/anatomy.json', '--nproc', 4, '--strict', int(strict)),
                   env(KNEE_THREADS=1), f'{W}/regions_log.txt')], strict)
    deadline = t0 + limit - n_all * 1.0 - 300
    run_parallel([(worker('cropping', '--work', W, '--nproc', 4, '--deadline', deadline, '--strict', int(strict)),
                   env(), f'{W}/crops_log.txt')], strict)
    print(f'crops: {len(glob.glob(f"{W}/crops/*.npz"))} of {n_all} studies | {(time.time() - t0) / 60:.1f} min', flush=True)


def run_models(a, W, t0, limit, run):
    blend = json.load(open(f'{a.models}/blend.json'))
    fams = []
    for name in blend:
        weights = sorted(glob.glob(f'{a.models}/{name}/*/best.pt'))
        if not weights:
            raise RuntimeError(f'no weights in {a.models}/{name}')
        fams.append(dict(name=name, config=f'{a.models}/{name}/config.json', weights=weights))
    cfgs = [json.load(open(f['config'])) for f in fams]
    names = cfgs[0]['crop_names']
    if any(c['crop_names'] != names for c in cfgs):
        raise RuntimeError('the model families were trained on different crops')
    rows = [dict(key=s, crops_file=f'{W}/crops/{s}.npz') for s in run['studies'].StudyInstanceUID]
    rows += [dict(key=f'stored|{s}', crops_file=f'{a.check_crops}/{s}.npz') for s in run['check_ids']]
    pd.DataFrame(rows).to_csv(f'{W}/meta/score.csv', index=False)
    job = dict(crop_names=names, min_fov=cfgs[0].get('min_fov', 0.25), k_eval=cfgs[0].get('K_eval', 12),
               studies=f'{W}/meta/score.csv', families=fams, out=W, workers=2)
    json.dump(job, open(f'{W}/meta/score_job.json', 'w'))
    n_gpu = torch.cuda.device_count()
    parts = max(1, n_gpu)
    run_parallel([(worker('scoring', '--job', f'{W}/meta/score_job.json', '--part', g, '--parts', parts, '--deadline', t0 + limit,
                          '--strict', int(run['strict'])), gpu_env(n_gpu, g), f'{W}/score_log{g}.txt') for g in range(parts)], run['strict'])
    keys = pd.read_csv(f'{W}/meta/score.csv', dtype=str).key.tolist()
    PR = {f['name']: np.full((len(keys), len(LABELS)), np.nan) for f in fams}
    for g in range(parts):
        p = f'{W}/scores_{g}of{parts}.npz'
        if not os.path.exists(p):
            if run['strict']:
                raise RuntimeError(f'missing model output {p}')
            continue
        z = np.load(p)
        for k in PR:
            PR[k][z['rows']] = z[k]
    num = sum(np.nan_to_num(PR[k]) * blend[k] * ~np.isnan(PR[k]) for k in PR)
    den = sum(blend[k] * ~np.isnan(PR[k]) for k in PR)
    P = np.full(num.shape, np.nan)
    np.divide(num, den, out=P, where=den > 0)
    print(f'models: {[f["name"] for f in fams]} | {(time.time() - t0) / 60:.1f} min', flush=True)
    return P, dict(zip(keys, range(len(keys))))


def write_submission(a, W, t0, run, P, at):
    order, strict = run['test_ids'], run['strict']
    rows = np.array([P[at[s]] if s in at else np.full(len(LABELS), np.nan) for s in order]).reshape(len(order), len(LABELS))
    ok = ~np.isnan(rows).any(1)
    if strict and not ok.all():
        raise RuntimeError(f'{int((~ok).sum())} test studies have no prediction: {[s for s, o in zip(order, ok) if not o][:5]}')
    if not ok.any():
        raise RuntimeError('no test study has a prediction')
    tset = set(order)
    RO = pd.read_csv(f'{W}/meta/rois.csv', dtype={'StudyInstanceUID': str})
    RO = RO[RO.StudyInstanceUID.isin(tset)]
    seg_err = set()
    for s in tset:
        f = f'{W}/seg/qc/{s}.json'
        if os.path.exists(f) and 'error' in [str(json.load(open(f)).get(k) or '') for k in ('flag_reasons', 'ref_flags', 'alt_flags')]:
            seg_err.add(s)
    crop_err = {s for s in tset if os.path.exists(f'{W}/crops/{s}.npz') and json.loads(str(np.load(f'{W}/crops/{s}.npz')['meta']))['errors']}
    SM = run['series']
    fb = {'no prediction': {s for s, o in zip(order, ok) if not o}, 'standard boxes': set(RO[RO.source != 'bones'].StudyInstanceUID),
          'unreadable series': set(SM[(SM.split == 'test') & SM.error.notna()].StudyInstanceUID), 'no sagittal': set(run['no_ref']) & tset,
          'segmentation errors': seg_err, 'crop errors': crop_err}
    fallback = set().union(*fb.values())
    cap = max(5, int(np.ceil(0.01 * len(order))))
    print(f'studies on a fallback path {len(fallback)} (limit {cap}) | ' + ' | '.join(f'{k} {len(v)}' for k, v in fb.items()), flush=True)
    if not strict and len(fallback) > cap:
        raise RuntimeError(f'{len(fallback)} of {len(order)} test studies needed a fallback (limit {cap}): '
                           + ', '.join(f'{k} {len(v)}' for k, v in fb.items()))
    rows = np.where(np.isnan(rows), np.nanmedian(rows[ok], 0)[None], rows)
    sub = pd.DataFrame(rows, columns=LABELS)
    sub.insert(0, 'StudyInstanceUID', order)
    sub.to_csv(a.out, index=False)
    print(f'{a.out}: {len(sub)} studies | predicted {int(ok.sum())} | filled {int((~ok).sum())} | '
          f'bone boxes {RO[RO.source == "bones"].StudyInstanceUID.nunique()} | {(time.time() - t0) / 60:.1f} min', flush=True)


def check_against_training(a, W, run, P, at):
    d_pred, d_pix, n_both, n_same, n_one = [], [], 0, 0, 0
    for s in run['check_ids']:
        d_pred.append(np.abs(P[at[s]] - P[at[f'stored|{s}']]))
        A1, m1 = crops.read_study(f'{W}/crops/{s}.npz')
        A2, m2 = crops.read_study(f'{a.check_crops}/{s}.npz')
        c1, c2 = set(m1['crops']), set(m2['crops'])
        n_one += len(c1 ^ c2)
        for c in c1 & c2:
            n_both += 1
            x, y = A1[f'{c}__img'], A2[f'{c}__img']
            if x.shape == y.shape:
                n_same += 1
                d_pix.append(float(np.abs(x.astype(np.int16) - y.astype(np.int16)).mean()))
    d = np.array(d_pred)
    print(f'check on {len(d)} training studies against their training crops: prediction difference mean {d.mean():.4f}, '
          f'max {d.max():.4f} | crops in both {n_both} (same size {n_same}), in one only {n_one} | '
          f'mean pixel difference {np.mean(d_pix) if d_pix else float("nan"):.2f} of 255', flush=True)
    if np.isnan(d).any() or d.mean() > 0.02:
        raise RuntimeError('the pipeline does not reproduce the training crops')


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument('--data', required=True)
    ap.add_argument('--models', required=True)
    ap.add_argument('--segmentation', required=True)
    ap.add_argument('--work', default='work')
    ap.add_argument('--out', default='submission.csv')
    ap.add_argument('--mode', choices=['auto', 'strict', 'lenient'], default='auto')
    ap.add_argument('--check-crops', default='')
    ap.add_argument('--hours', type=float, default=8.6)
    a = ap.parse_args(argv)
    t0, limit = time.time(), a.hours * 3600
    W = os.path.abspath(a.work)
    for d in (W, f'{W}/meta', f'{W}/seg', f'{W}/nnunet'):
        os.makedirs(d, exist_ok=True)
    run = find_studies(a, W, t0)
    run_preprocessing(a, W, t0, limit, run)
    P, at = run_models(a, W, t0, limit, run)
    write_submission(a, W, t0, run, P, at)
    if run['check_ids']:
        check_against_training(a, W, run, P, at)


if __name__ == '__main__':
    main()

In [ ]:
sys.path.insert(0, CODE)
import predict
args = ['--data', data, '--models', f'{assets}/models', '--segmentation', seg[0], '--work', '/kaggle/tmp/work',
        '--out', '/kaggle/working/submission.csv', '--hours', '8.5']
predict.main(args + (['--check-crops', check] if check else []))